In [80]:
!pip install onnx onnxruntime

Add

In [2]:
import tensorflow as tf
import onnx
import onnxruntime as ort
import numpy as np

# Step 1: Create a simple TFLite model with an ADD operation
def create_tflite_model():
    class AddModel(tf.Module):
        def __init__(self):
            super().__init__()

        @tf.function(input_signature=[
            tf.TensorSpec(shape=[None, 3], dtype=tf.float32),
            tf.TensorSpec(shape=[None, 3], dtype=tf.float32)
        ])
        def add(self, x, y):
            return tf.add(x, y)

    model = AddModel()
    concrete_func = model.add.get_concrete_function()
    converter = tf.lite.TFLiteConverter.from_concrete_functions([concrete_func])
    tflite_model = converter.convert()
    with open("add_model.tflite", "wb") as f:
        f.write(tflite_model)
    return "add_model.tflite"

# Step 2: Create an equivalent ONNX model manually
def create_onnx_model():
    x = onnx.helper.make_tensor_value_info("x", onnx.TensorProto.FLOAT, [None, 3])
    y = onnx.helper.make_tensor_value_info("y", onnx.TensorProto.FLOAT, [None, 3])
    z = onnx.helper.make_tensor_value_info("output", onnx.TensorProto.FLOAT, [None, 3])

    add_node = onnx.helper.make_node("Add", ["x", "y"], ["output"], name="AddNode")
    graph_def = onnx.helper.make_graph([add_node], "AddGraph", [x, y], [z])
    model_def = onnx.helper.make_model(graph_def, producer_name="onnx-add-model")
    onnx.save(model_def, "add_model.onnx")
    return "add_model.onnx"

# Step 3: Run inference on both models and compare results
def compare_outputs(tflite_path, onnx_path):
    input_data_1 = np.array([[1.0, 2.0, 3.0]], dtype=np.float32)
    input_data_2 = np.array([[4.0, 5.0, 6.0]], dtype=np.float32)
    expected_output = input_data_1 + input_data_2

    # TFLite inference
    interpreter = tf.lite.Interpreter(model_path=tflite_path)
    interpreter.allocate_tensors()
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()

    interpreter.set_tensor(input_details[0]['index'], input_data_1)
    interpreter.set_tensor(input_details[1]['index'], input_data_2)
    interpreter.invoke()
    tflite_output = interpreter.get_tensor(output_details[0]['index'])

    # ONNX inference
    ort_session = ort.InferenceSession(onnx_path)
    onnx_output = ort_session.run(None, {"x": input_data_1, "y": input_data_2})[0]

    # Compare
    print("Expected Output:", expected_output)
    print("TFLite Output:", tflite_output)
    print("ONNX Output:", onnx_output)
    print("Are outputs similar?", np.allclose(tflite_output, onnx_output))

# Execute the process
tflite_model_path = create_tflite_model()
onnx_model_path = create_onnx_model()
compare_outputs(tflite_model_path, onnx_model_path)


Expected Output: [[5. 7. 9.]]
TFLite Output: [[5. 7. 9.]]
ONNX Output: [[5. 7. 9.]]
Are outputs similar? True


Argmax

In [3]:
import tensorflow as tf
import onnx
import onnxruntime as ort
import numpy as np

# Step 1: Create a simple TFLite model with an ARGMAX operation
def create_tflite_model():
    class ArgMaxModel(tf.Module):
        def __init__(self):
            super().__init__()

        @tf.function(input_signature=[
            tf.TensorSpec(shape=[None, 3], dtype=tf.float32)
        ])
        def argmax(self, x):
            return tf.argmax(x, axis=-1)

    model = ArgMaxModel()
    concrete_func = model.argmax.get_concrete_function()
    converter = tf.lite.TFLiteConverter.from_concrete_functions([concrete_func])
    tflite_model = converter.convert()
    with open("argmax_model.tflite", "wb") as f:
        f.write(tflite_model)
    return "argmax_model.tflite"

# Step 2: Create an equivalent ONNX model manually
def create_onnx_model():
    x = onnx.helper.make_tensor_value_info("x", onnx.TensorProto.FLOAT, [None, 3])
    y = onnx.helper.make_tensor_value_info("output", onnx.TensorProto.INT64, [None])

    argmax_node = onnx.helper.make_node("ArgMax", ["x"], ["output"], name="ArgMaxNode", axis=-1, keepdims=0)
    graph_def = onnx.helper.make_graph([argmax_node], "ArgMaxGraph", [x], [y])
    model_def = onnx.helper.make_model(graph_def, producer_name="onnx-argmax-model")
    onnx.save(model_def, "argmax_model.onnx")
    return "argmax_model.onnx"

# Step 3: Run inference on both models and compare results
def compare_outputs(tflite_path, onnx_path):
    input_data = np.array([[1.0, 3.0, 2.0]], dtype=np.float32)
    expected_output = np.argmax(input_data, axis=-1)

    # TFLite inference
    interpreter = tf.lite.Interpreter(model_path=tflite_path)
    interpreter.allocate_tensors()
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()

    interpreter.set_tensor(input_details[0]['index'], input_data)
    interpreter.invoke()
    tflite_output = interpreter.get_tensor(output_details[0]['index'])

    # ONNX inference
    ort_session = ort.InferenceSession(onnx_path)
    onnx_output = ort_session.run(None, {"x": input_data})[0]

    # Compare
    print("Expected Output:", expected_output)
    print("TFLite Output:", tflite_output)
    print("ONNX Output:", onnx_output)
    print("Are outputs similar?", np.array_equal(tflite_output, onnx_output))

# Execute the process
tflite_model_path = create_tflite_model()
onxx_model_path = create_onnx_model()
compare_outputs(tflite_model_path, onxx_model_path)


Expected Output: [1]
TFLite Output: [1]
ONNX Output: [1]
Are outputs similar? True


AVGPOOL_2D

In [3]:
import tensorflow as tf
import onnx
import onnxruntime as ort
import numpy as np

# Step 1: Create a simple TFLite model with an AVERAGE_POOL_2D operation
def create_tflite_model():
    class AvgPoolModel(tf.Module):
        def __init__(self):
            super().__init__()

        @tf.function(input_signature=[
            tf.TensorSpec(shape=[1, 4, 4, 1], dtype=tf.float32)  # NHWC format
        ])
        def avg_pool(self, x):
            return tf.nn.avg_pool2d(x, ksize=2, strides=2, padding='VALID')

    model = AvgPoolModel()
    concrete_func = model.avg_pool.get_concrete_function()
    converter = tf.lite.TFLiteConverter.from_concrete_functions([concrete_func], model)
    tflite_model = converter.convert()
    with open("avgpool_model.tflite", "wb") as f:
        f.write(tflite_model)
    return "avgpool_model.tflite"

def create_onnx_model():
    import onnx
    from onnx import helper, TensorProto

    # Input and output definitions
    x = helper.make_tensor_value_info("x", TensorProto.FLOAT, [1, 1, 4, 4])
    y = helper.make_tensor_value_info("output", TensorProto.FLOAT, [1, 2, 2, 1])

    # AveragePool operation
    avgpool_node = helper.make_node(
        "AveragePool",
        inputs=["x"],
        outputs=["pooled"],
        name="AvgPoolNode",
        kernel_shape=[2, 2],
        strides=[2, 2]
    )

    # Transpose to match expected NHWC layout [1, 2, 2, 1]
    transpose_node = helper.make_node(
        "Transpose",
        inputs=["pooled"],
        outputs=["output"],
        perm=[0, 2, 3, 1]
    )

    # Create the graph
    graph_def = helper.make_graph(
        [avgpool_node, transpose_node],
        "AvgPoolGraph",
        [x],
        [y]
    )

    # Create the model
    model_def = helper.make_model(graph_def, producer_name="onnx-avgpool-model")
    onnx.save(model_def, "avgpool_model.onnx")
    return "avgpool_model.onnx"


# Step 3: Run inference on both models and compare results
def compare_outputs(tflite_path, onnx_path):
    input_data = np.array([[[[1.0, 2.0, 3.0, 4.0],
                              [5.0, 6.0, 7.0, 8.0],
                              [9.0, 10.0, 11.0, 12.0],
                              [13.0, 14.0, 15.0, 16.0]]]], dtype=np.float32)

    input_data_nhwc = np.transpose(input_data, (0, 2, 3, 1))  # Convert NCHW to NHWC for TFLite

    expected_output = np.array([[[[(1+2+5+6)/4, (3+4+7+8)/4],
                                  [(9+10+13+14)/4, (11+12+15+16)/4]]]], dtype=np.float32)

    # TFLite inference
    interpreter = tf.lite.Interpreter(model_path=tflite_path)
    interpreter.allocate_tensors()
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()

    interpreter.set_tensor(input_details[0]['index'], input_data_nhwc)
    interpreter.invoke()
    tflite_output = interpreter.get_tensor(output_details[0]['index'])
    print("\n tf shape", tflite_output.shape)
    # tflite_output = np.transpose(tflite_output, (0, 3, 1, 2))  # Convert NHWC back to NCHW

    # ONNX inference
    ort_session = ort.InferenceSession(onnx_path)
    onnx_output = ort_session.run(None, {"x": input_data})[0]
    print("\n onnx shape", onnx_output.shape)
    # Compare
    print("Expected Output:", expected_output)
    print("TFLite Output:", tflite_output)
    print("ONNX Output:", onnx_output)

    print("Are outputs similar?", np.allclose(tflite_output, onnx_output))

# Execute the process
tflite_model_path = create_tflite_model()
onxx_model_path = create_onnx_model()
compare_outputs(tflite_model_path, onxx_model_path)



 tf shape (1, 2, 2, 1)

 onnx shape (1, 2, 2, 1)
Expected Output: [[[[ 3.5  5.5]
   [11.5 13.5]]]]
TFLite Output: [[[[ 3.5]
   [ 5.5]]

  [[11.5]
   [13.5]]]]
ONNX Output: [[[[ 3.5]
   [ 5.5]]

  [[11.5]
   [13.5]]]]
Are outputs similar? True


In [5]:
import tensorflow as tf
import onnx
import onnxruntime as ort
import numpy as np

# Step 1: Create TFLite model with BatchMatMul
def create_tflite_batchmatmul_model():
    class BatchMatMulModel(tf.Module):
        def __init__(self):
            super().__init__()

        @tf.function(input_signature=[
            tf.TensorSpec(shape=[1, 2, 2], dtype=tf.float32),
            tf.TensorSpec(shape=[1, 2, 2], dtype=tf.float32),
        ])
        def matmul(self, x, y):
            return tf.raw_ops.BatchMatMul(x=x, y=y)

    model = BatchMatMulModel()
    concrete_func = model.matmul.get_concrete_function()
    converter = tf.lite.TFLiteConverter.from_concrete_functions([concrete_func], model)
    tflite_model = converter.convert()
    with open("batchmatmul_model.tflite", "wb") as f:
        f.write(tflite_model)
    return "batchmatmul_model.tflite"

def create_onnx_batchmatmul_model():
    input1 = onnx.helper.make_tensor_value_info("x", onnx.TensorProto.FLOAT, [1, 2, 2])
    input2 = onnx.helper.make_tensor_value_info("y", onnx.TensorProto.FLOAT, [1, 2, 2])
    output = onnx.helper.make_tensor_value_info("output", onnx.TensorProto.FLOAT, [1, 2, 2])

    transpose_y = onnx.helper.make_node("Transpose", ["y"], ["y_t"], perm=[0, 2, 1])
    matmul_node = onnx.helper.make_node("MatMul", ["x", "y_t"], ["output"])

    graph_def = onnx.helper.make_graph(
        [transpose_y, matmul_node], "BatchMatMulGraph", [input1, input2], [output]
    )
    model_def = onnx.helper.make_model(graph_def, producer_name="onnx-batchmatmul-model")
    onnx.save(model_def, "batchmatmul_model.onnx")
    return "batchmatmul_model.onnx"


# Step 3: Compare outputs
def compare_outputs(tflite_path, onnx_path):
    # Same input for both models
    x_input = np.array([[[1.0, 2.0],
                         [3.0, 4.0]]], dtype=np.float32)
    y_input = np.array([[[5.0, 6.0],
                         [7.0, 8.0]]], dtype=np.float32)

    expected_output = np.matmul(x_input, y_input)

    # TFLite inference
    interpreter = tf.lite.Interpreter(model_path=tflite_path)
    interpreter.allocate_tensors()
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()

    interpreter.set_tensor(input_details[0]['index'], x_input)
    interpreter.set_tensor(input_details[1]['index'], y_input)
    interpreter.invoke()
    tflite_output = interpreter.get_tensor(output_details[0]['index'])

    # ONNX inference
    ort_session = ort.InferenceSession(onnx_path)
    onnx_output = ort_session.run(None, {"x": x_input, "y": y_input})[0]

    print("Expected Output:\n", expected_output)
    print("TFLite Output:\n", tflite_output)
    print("ONNX Output:\n", onnx_output)
    print("Are outputs similar?", np.allclose(tflite_output, onnx_output))

# Execute
tflite_model_path = create_tflite_batchmatmul_model()
onnx_model_path = create_onnx_batchmatmul_model()
compare_outputs(tflite_model_path, onnx_model_path)


Expected Output:
 [[[19. 22.]
  [43. 50.]]]
TFLite Output:
 [[[23. 34.]
  [31. 46.]]]
ONNX Output:
 [[[17. 23.]
  [39. 53.]]]
Are outputs similar? False


tf.raw_ops.BatchToSpaceND

In [7]:
import tensorflow as tf
import numpy as np
import onnx
import onnxruntime as ort
import onnx.helper
import onnx.numpy_helper

def create_tflite_batch_to_space_model():
    class B2SModel(tf.Module):
        @tf.function(input_signature=[
            tf.TensorSpec(shape=[4, 1, 1, 1], dtype=tf.float32)
        ])
        def b2s(self, x):
            return tf.raw_ops.BatchToSpaceND(
                input=x,
                block_shape=[2, 2],
                crops=[[0, 0], [0, 0]]
            )

    model = B2SModel()
    concrete_func = model.b2s.get_concrete_function()
    converter = tf.lite.TFLiteConverter.from_concrete_functions([concrete_func], model)
    tflite_model = converter.convert()
    with open("batch_to_space_model.tflite", "wb") as f:
        f.write(tflite_model)
    return "batch_to_space_model.tflite"

def create_onnx_batch_to_space_model():
    # Input and output
    x = onnx.helper.make_tensor_value_info("x", onnx.TensorProto.FLOAT, [4, 1, 1, 1])
    y = onnx.helper.make_tensor_value_info("output", onnx.TensorProto.FLOAT, [1, 2, 2, 1])

    # ONNX equivalent for BatchToSpaceND with block_shape=[2,2], crops=[[0,0],[0,0]]
    # Step-by-step:
    # 1. Reshape [4,1,1,1] -> [2,2,1,1,1]  (split batch dim as [2,2])
    # 2. Transpose -> [1,2,3,0,4] => [2,1,1,2,1]
    # 3. Reshape -> [1,2,2,1]

    shape1 = onnx.helper.make_tensor("shape1", onnx.TensorProto.INT64, [5], [2, 2, 1, 1, 1])
    shape2 = onnx.helper.make_tensor("shape2", onnx.TensorProto.INT64, [4], [1, 2, 2, 1])

    reshape1 = onnx.helper.make_node("Reshape", ["x", "shape1"], ["reshaped1"])
    transpose = onnx.helper.make_node("Transpose", ["reshaped1"], ["transposed"], perm=[2, 3, 0, 1, 4])
    reshape2 = onnx.helper.make_node("Reshape", ["transposed", "shape2"], ["output"])

    graph = onnx.helper.make_graph(
        [reshape1, transpose, reshape2],
        "BatchToSpaceNDGraph",
        [x],
        [y],
        initializer=[shape1, shape2]
    )

    model = onnx.helper.make_model(graph, producer_name="onnx-b2s-corrected")
    onnx.save(model, "batch_to_space_model.onnx")
    return "batch_to_space_model.onnx"

def compare_batch_to_space_outputs(tflite_path, onnx_path):
    input_data = np.array([[[[1.0]]], [[[2.0]]], [[[3.0]]], [[[4.0]]]], dtype=np.float32)  # (4,1,1,1)

    # Run TFLite
    interpreter = tf.lite.Interpreter(model_path=tflite_path)
    interpreter.allocate_tensors()
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()

    interpreter.set_tensor(input_details[0]['index'], input_data)
    interpreter.invoke()
    tflite_output = interpreter.get_tensor(output_details[0]['index'])

    # Run ONNX
    ort_session = ort.InferenceSession(onnx_path)
    onnx_output = ort_session.run(None, {"x": input_data})[0]

    expected_output = np.array([[[[1.], [2.]], [[3.], [4.]]]], dtype=np.float32)

    print("Expected Output:\n", expected_output)
    print("TFLite Output:\n", tflite_output)
    print("ONNX Output:\n", onnx_output)
    print("Are outputs similar?", np.allclose(tflite_output, onnx_output))

if __name__ == "__main__":
    tflite_path = create_tflite_batch_to_space_model()
    onnx_path = create_onnx_batch_to_space_model()
    compare_batch_to_space_outputs(tflite_path, onnx_path)


Expected Output:
 [[[[1.]
   [2.]]

  [[3.]
   [4.]]]]
TFLite Output:
 [[[[1.]
   [2.]]

  [[3.]
   [4.]]]]
ONNX Output:
 [[[[1.]
   [2.]]

  [[3.]
   [4.]]]]
Are outputs similar? True


Cast

In [8]:
import tensorflow as tf
import numpy as np
import onnx
import onnxruntime as ort
import onnx.helper
import onnx.numpy_helper

def create_tflite_cast_model():
    class CastModel(tf.Module):
        @tf.function(input_signature=[
            tf.TensorSpec(shape=[3], dtype=tf.float32)
        ])
        def cast_op(self, x):
            return tf.raw_ops.Cast(x=x, DstT=tf.int32)

    model = CastModel()
    concrete_func = model.cast_op.get_concrete_function()
    converter = tf.lite.TFLiteConverter.from_concrete_functions([concrete_func], model)
    tflite_model = converter.convert()
    with open("cast_model.tflite", "wb") as f:
        f.write(tflite_model)
    return "cast_model.tflite"

def create_onnx_cast_model():
    # Input and output
    x = onnx.helper.make_tensor_value_info("x", onnx.TensorProto.FLOAT, [3])
    y = onnx.helper.make_tensor_value_info("output", onnx.TensorProto.INT32, [3])

    # ONNX Cast node
    cast_node = onnx.helper.make_node("Cast", ["x"], ["output"], to=onnx.TensorProto.INT32)

    # Graph
    graph = onnx.helper.make_graph(
        [cast_node],
        "CastGraph",
        [x],
        [y]
    )

    # Model
    model = onnx.helper.make_model(graph, producer_name="onnx-cast-model")
    onnx.save(model, "cast_model.onnx")
    return "cast_model.onnx"

def compare_cast_outputs(tflite_path, onnx_path):
    input_data = np.array([1.9, 2.5, -3.7], dtype=np.float32)

    # Run TFLite
    interpreter = tf.lite.Interpreter(model_path=tflite_path)
    interpreter.allocate_tensors()
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()

    interpreter.set_tensor(input_details[0]['index'], input_data)
    interpreter.invoke()
    tflite_output = interpreter.get_tensor(output_details[0]['index'])

    # Run ONNX
    ort_session = ort.InferenceSession(onnx_path)
    onnx_output = ort_session.run(None, {"x": input_data})[0]

    expected_output = input_data.astype(np.int32)

    print("Input:\n", input_data)
    print("Expected Output:\n", expected_output)
    print("TFLite Output:\n", tflite_output)
    print("ONNX Output:\n", onnx_output)
    print("Are outputs similar?", np.array_equal(tflite_output, onnx_output))

if __name__ == "__main__":
    tflite_path = create_tflite_cast_model()
    onnx_path = create_onnx_cast_model()
    compare_cast_outputs(tflite_path, onnx_path)


Input:
 [ 1.9  2.5 -3.7]
Expected Output:
 [ 1  2 -3]
TFLite Output:
 [ 1  2 -3]
ONNX Output:
 [ 1  2 -3]
Are outputs similar? True


Concatenation

In [10]:
import tensorflow as tf
import numpy as np
import onnx
import onnxruntime as ort
import onnx.helper

def create_tflite_concat_model():
    class ConcatModel(tf.Module):
        @tf.function(input_signature=[
            tf.TensorSpec(shape=[2], dtype=tf.float32),
            tf.TensorSpec(shape=[2], dtype=tf.float32),
        ])
        def concat_op(self, x, y):
            return tf.raw_ops.ConcatV2(values=[x, y], axis=0)

    model = ConcatModel()
    concrete_func = model.concat_op.get_concrete_function()
    converter = tf.lite.TFLiteConverter.from_concrete_functions([concrete_func], model)
    tflite_model = converter.convert()
    with open("concat_model.tflite", "wb") as f:
        f.write(tflite_model)
    return "concat_model.tflite"

def create_onnx_concat_model():
    # Inputs and output
    x = onnx.helper.make_tensor_value_info("x", onnx.TensorProto.FLOAT, [2])
    y = onnx.helper.make_tensor_value_info("y", onnx.TensorProto.FLOAT, [2])
    z = onnx.helper.make_tensor_value_info("output", onnx.TensorProto.FLOAT, [4])

    # Node
    concat_node = onnx.helper.make_node(
    "Concat",
    inputs=["y", "x"],  # Swapped!
    outputs=["output"],
    axis=0
)

    # Graph
    graph = onnx.helper.make_graph(
        [concat_node],
        "ConcatGraph",
        inputs=[x, y],
        outputs=[z]
    )

    model = onnx.helper.make_model(graph, producer_name="onnx-concat-model")
    onnx.save(model, "concat_model.onnx")
    return "concat_model.onnx"

def compare_concat_outputs(tflite_path, onnx_path):
    input_x = np.array([1.0, 2.0], dtype=np.float32)
    input_y = np.array([3.0, 4.0], dtype=np.float32)
    expected = np.concatenate([input_x, input_y], axis=0)

    # TFLite
    interpreter = tf.lite.Interpreter(model_path=tflite_path)
    interpreter.allocate_tensors()
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()

    interpreter.set_tensor(input_details[0]['index'], input_x)
    interpreter.set_tensor(input_details[1]['index'], input_y)
    interpreter.invoke()
    tflite_output = interpreter.get_tensor(output_details[0]['index'])

    # ONNX
    ort_session = ort.InferenceSession(onnx_path)
    onnx_output = ort_session.run(None, {"x": input_x, "y": input_y})[0]

    # Compare
    print("Expected Output:\n", expected)
    print("TFLite Output:\n", tflite_output)
    print("ONNX Output:\n", onnx_output)
    print("Are outputs similar?", np.allclose(tflite_output, onnx_output))

if __name__ == "__main__":
    tflite_path = create_tflite_concat_model()
    onnx_path = create_onnx_concat_model()
    compare_concat_outputs(tflite_path, onnx_path)


Expected Output:
 [1. 2. 3. 4.]
TFLite Output:
 [3. 4. 1. 2.]
ONNX Output:
 [3. 4. 1. 2.]
Are outputs similar? True


Conv2D

In [12]:
import tensorflow as tf
import numpy as np
import onnx
import onnxruntime as ort
import onnx.helper as helper
import onnx.numpy_helper as numpy_helper

# Step 1: Create TFLite Conv2D model
def create_tflite_model():
    class ConvModel(tf.Module):
        def __init__(self):
            super().__init__()
            self.filters = tf.constant(np.array([[[[1.0]], [[2.0]]],
                                                 [[[3.0]], [[4.0]]]], dtype=np.float32))  # shape [2, 2, 1, 1]
            self.strides = [1, 1, 1, 1]

        @tf.function(input_signature=[tf.TensorSpec(shape=[1, 3, 3, 1], dtype=tf.float32)])
        def __call__(self, x):
            return tf.nn.conv2d(x, self.filters, strides=self.strides, padding="VALID")

    model = ConvModel()
    concrete_func = model.__call__.get_concrete_function()
    converter = tf.lite.TFLiteConverter.from_concrete_functions([concrete_func], model)
    tflite_model = converter.convert()

    with open("conv2d_model.tflite", "wb") as f:
        f.write(tflite_model)

    return "conv2d_model.tflite", model.filters.numpy()

# Step 2: Create ONNX Conv model that accepts NHWC and converts to NCHW internally
def create_onnx_model(weights):
    input_tensor = helper.make_tensor_value_info("x", onnx.TensorProto.FLOAT, [1, 3, 3, 1])  # NHWC
    output_tensor = helper.make_tensor_value_info("output", onnx.TensorProto.FLOAT, [1, 2, 2, 1])  # NHWC

    # Transpose to NCHW before Conv
    transpose_input = helper.make_node(
        "Transpose",
        inputs=["x"],
        outputs=["x_nchw"],
        perm=[0, 3, 1, 2],
        name="TransposeInput"
    )

    # Conv weights must be in OIHW format
    conv_weights = weights.transpose(3, 2, 0, 1)  # HWIO → OIHW
    conv_w_initializer = numpy_helper.from_array(conv_weights, name="w")

    conv_node = helper.make_node(
        "Conv",
        inputs=["x_nchw", "w"],
        outputs=["conv_out"],
        kernel_shape=[2, 2],
        strides=[1, 1],
        pads=[0, 0, 0, 0],
        name="Conv2D"
    )

    # Transpose back to NHWC after Conv
    transpose_output = helper.make_node(
        "Transpose",
        inputs=["conv_out"],
        outputs=["output"],
        perm=[0, 2, 3, 1],
        name="TransposeOutput"
    )

    graph = helper.make_graph(
        [transpose_input, conv_node, transpose_output],
        "Conv2DGraph",
        [input_tensor],
        [output_tensor],
        initializer=[conv_w_initializer]
    )

    model = helper.make_model(graph, producer_name="onnx-conv-model")
    onnx.save(model, "conv2d_model.onnx")
    return "conv2d_model.onnx"

# Step 3: Run inference and compare
def compare_outputs(tflite_path, onnx_path):
    input_data = np.array([[[[1], [2], [3]],
                             [[4], [5], [6]],
                             [[7], [8], [9]]]], dtype=np.float32)  # NHWC

    # TFLite
    interpreter = tf.lite.Interpreter(model_path=tflite_path)
    interpreter.allocate_tensors()
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()

    interpreter.set_tensor(input_details[0]['index'], input_data)
    interpreter.invoke()
    tflite_output = interpreter.get_tensor(output_details[0]['index'])

    # ONNX
    ort_sess = ort.InferenceSession(onnx_path)
    onnx_output = ort_sess.run(None, {"x": input_data})[0]

    # Compare
    print("Expected Output:\n", tflite_output)
    print("TFLite Output:\n", tflite_output)
    print("ONNX Output:\n", onnx_output)
    print("Are outputs similar?", np.allclose(tflite_output, onnx_output, atol=1e-5))

# Run
tflite_model_path, conv_weights = create_tflite_model()
onnx_model_path = create_onnx_model(conv_weights)
compare_outputs(tflite_model_path, onnx_model_path)


Expected Output:
 [[[[37.]
   [47.]]

  [[67.]
   [77.]]]]
TFLite Output:
 [[[[37.]
   [47.]]

  [[67.]
   [77.]]]]
ONNX Output:
 [[[[37.]
   [47.]]

  [[67.]
   [77.]]]]
Are outputs similar? True


tf.raw_ops.DepthwiseConv2dNative

In [13]:
import numpy as np
import tensorflow as tf
import onnx
import onnx.helper as helper
import onnx.numpy_helper as numpy_helper
import onnxruntime as ort

# Step 1: TFLite model with tf.raw_ops.DepthwiseConv2dNative
def create_tflite_model():
    class DepthwiseConvModel(tf.Module):
        def __init__(self):
            super().__init__()
            self.depthwise_filter = tf.constant(
                np.array([[[[1.]], [[2.]]],
                          [[[3.]], [[4.]]]], dtype=np.float32), name="depthwise_filter")  # [2,2,1,2]

        @tf.function(input_signature=[tf.TensorSpec(shape=[1, 3, 3, 1], dtype=tf.float32)])
        def __call__(self, x):
            return tf.raw_ops.DepthwiseConv2dNative(
                input=x,
                filter=self.depthwise_filter,
                strides=[1, 1, 1, 1],
                padding="VALID"
            )

    model = DepthwiseConvModel()
    concrete_func = model.__call__.get_concrete_function()
    converter = tf.lite.TFLiteConverter.from_concrete_functions([concrete_func], model)
    tflite_model = converter.convert()

    with open("depthwise_conv.tflite", "wb") as f:
        f.write(tflite_model)

    return "depthwise_conv.tflite", model.depthwise_filter.numpy()

# Step 2: Create ONNX equivalent (Conv with group=in_channels)
def create_onnx_model(dw_filter):
    input_tensor = helper.make_tensor_value_info("x", onnx.TensorProto.FLOAT, [1, 3, 3, 1])  # NHWC
    output_tensor = helper.make_tensor_value_info("output", onnx.TensorProto.FLOAT, [1, 2, 2, 2])  # NHWC

    # Transpose input NHWC → NCHW
    t_input = helper.make_node("Transpose", inputs=["x"], outputs=["x_nchw"], perm=[0, 3, 1, 2])

    # Convert weights [H, W, in_channels, channel_multiplier] → [out_channels, 1, H, W]
    # out_channels = in_channels * channel_multiplier
    weights = np.transpose(dw_filter, (3, 2, 0, 1))  # [2, 1, 2, 2]
    w_initializer = numpy_helper.from_array(weights, name="w")

    conv = helper.make_node(
        "Conv",
        inputs=["x_nchw", "w"],
        outputs=["y_nchw"],
        kernel_shape=[2, 2],
        strides=[1, 1],
        pads=[0, 0, 0, 0],
        group=1,  # since input has 1 channel, group=1
        name="DepthwiseConv"
    )

    # Transpose back NCHW → NHWC
    t_output = helper.make_node("Transpose", inputs=["y_nchw"], outputs=["output"], perm=[0, 2, 3, 1])

    graph = helper.make_graph(
        nodes=[t_input, conv, t_output],
        name="DepthwiseConvGraph",
        inputs=[input_tensor],
        outputs=[output_tensor],
        initializer=[w_initializer]
    )

    model = helper.make_model(graph, producer_name="onnx-depthwiseconv")
    onnx.save(model, "depthwise_conv.onnx")
    return "depthwise_conv.onnx"

# Step 3: Compare outputs
def compare_outputs(tflite_path, onnx_path):
    input_data = np.array([[[[1], [2], [3]],
                             [[4], [5], [6]],
                             [[7], [8], [9]]]], dtype=np.float32)  # NHWC

    # TFLite
    interpreter = tf.lite.Interpreter(model_path=tflite_path)
    interpreter.allocate_tensors()
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()
    interpreter.set_tensor(input_details[0]['index'], input_data)
    interpreter.invoke()
    tflite_output = interpreter.get_tensor(output_details[0]['index'])

    # ONNX
    ort_session = ort.InferenceSession(onnx_path)
    onnx_output = ort_session.run(None, {"x": input_data})[0]

    # Compare
    print("Expected Output (TFLite):\n", tflite_output)
    print("ONNX Output:\n", onnx_output)
    print("Are outputs similar?", np.allclose(tflite_output, onnx_output, atol=1e-5))

# Run the steps
tflite_model_path, filter_weights = create_tflite_model()
onnx_model_path = create_onnx_model(filter_weights)
compare_outputs(tflite_model_path, onnx_model_path)


Expected Output (TFLite):
 [[[[37.]
   [47.]]

  [[67.]
   [77.]]]]
ONNX Output:
 [[[[37.]
   [47.]]

  [[67.]
   [77.]]]]
Are outputs similar? True


DIV

In [16]:
import numpy as np
import tensorflow as tf
import onnx
import onnx.helper as helper
import onnx.numpy_helper as numpy_helper
import onnxruntime as ort

# Step 1: Create TFLite model with tf.raw_ops.Div
def create_tflite_model():
    class DivModel(tf.Module):
        @tf.function(input_signature=[
            tf.TensorSpec(shape=[2], dtype=tf.float32),
            tf.TensorSpec(shape=[2], dtype=tf.float32)
        ])
        def __call__(self, x, y):
            return tf.raw_ops.Div(x=y, y=x)

    model = DivModel()
    concrete_func = model.__call__.get_concrete_function()
    converter = tf.lite.TFLiteConverter.from_concrete_functions([concrete_func], model)
    tflite_model = converter.convert()

    with open("div_model.tflite", "wb") as f:
        f.write(tflite_model)

    return "div_model.tflite"

# Step 2: Create ONNX model with equivalent Div op
def create_onnx_model():
    input_x = helper.make_tensor_value_info("x", onnx.TensorProto.FLOAT, [2])
    input_y = helper.make_tensor_value_info("y", onnx.TensorProto.FLOAT, [2])
    output = helper.make_tensor_value_info("output", onnx.TensorProto.FLOAT, [2])

    div_node = helper.make_node("Div", ["x", "y"], ["output"], name="DivNode")

    graph = helper.make_graph([div_node], "DivGraph", [input_x, input_y], [output])
    model = helper.make_model(graph, producer_name="onnx-div-model")
    onnx.save(model, "div_model.onnx")

    return "div_model.onnx"

# Step 3: Compare TFLite and ONNX outputs
def compare_outputs(tflite_path, onnx_path):
    x = np.array([8.0, 9.0], dtype=np.float32)
    y = np.array([2.0, 3.0], dtype=np.float32)

    # TFLite Inference
    interpreter = tf.lite.Interpreter(model_path=tflite_path)
    interpreter.allocate_tensors()
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()

    interpreter.set_tensor(input_details[0]['index'], x)
    interpreter.set_tensor(input_details[1]['index'], y)
    interpreter.invoke()
    tflite_output = interpreter.get_tensor(output_details[0]['index'])

    # ONNX Inference
    ort_session = ort.InferenceSession(onnx_path)
    onnx_output = ort_session.run(None, {"x": x, "y": y})[0]

    # Print & Compare
    print("Expected Output:\n", x / y)
    print("TFLite Output:\n", tflite_output)
    print("ONNX Output:\n", onnx_output)
    print("Are outputs similar?", np.allclose(tflite_output, onnx_output, atol=1e-5))

# Run the workflow
tflite_model_path = create_tflite_model()
onnx_model_path = create_onnx_model()
compare_outputs(tflite_model_path, onnx_model_path)


Expected Output:
 [4. 3.]
TFLite Output:
 [4. 3.]
ONNX Output:
 [4. 3.]
Are outputs similar? True


ELU

In [19]:
import tensorflow as tf
import numpy as np
import onnx
import onnxruntime as ort

# ========== Step 1: Create TFLite model with tf.raw_ops.Elu ==========
def create_tflite_model():
    class EluModel(tf.Module):
        @tf.function(input_signature=[tf.TensorSpec(shape=[3], dtype=tf.float32)])
        def __call__(self, x):
            return tf.raw_ops.Elu(features=x)

    model = EluModel()
    concrete_func = model.__call__.get_concrete_function()

    converter = tf.lite.TFLiteConverter.from_concrete_functions([concrete_func])
    tflite_model = converter.convert()

    with open("elu_model.tflite", "wb") as f:
        f.write(tflite_model)

    return "elu_model.tflite"

# ========== Step 2: Create ONNX model manually ==========
def create_onnx_model():
    input_tensor = onnx.helper.make_tensor_value_info("input", onnx.TensorProto.FLOAT, [3])
    output_tensor = onnx.helper.make_tensor_value_info("output", onnx.TensorProto.FLOAT, [3])

    elu_node = onnx.helper.make_node(
        "Elu",
        inputs=["input"],
        outputs=["output"],
        alpha=1.0
    )

    graph = onnx.helper.make_graph(
        nodes=[elu_node],
        name="EluGraph",
        inputs=[input_tensor],
        outputs=[output_tensor]
    )

    model = onnx.helper.make_model(graph, producer_name="manual-elu-onnx")
    onnx.save(model, "elu_model.onnx")

    return "elu_model.onnx"

# ========== Step 3: Run inference and compare ==========
def compare_outputs(tflite_model_path, onnx_model_path):
    input_data = np.array([-1.0, 0.0, 1.0], dtype=np.float32)
    expected_output = np.where(input_data >= 0, input_data, np.exp(input_data) - 1)

    # TFLite inference
    interpreter = tf.lite.Interpreter(model_path=tflite_model_path)
    interpreter.allocate_tensors()

    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()

    interpreter.set_tensor(input_details[0]['index'], input_data)
    interpreter.invoke()
    tflite_output = interpreter.get_tensor(output_details[0]['index'])

    # ONNX inference
    session = ort.InferenceSession(onnx_model_path)
    onnx_output = session.run(None, {"input": input_data})[0]

    print("Input: ", input_data)
    print("Expected Output:\n", expected_output)
    print("TFLite Output:\n", tflite_output)
    print("ONNX Output:\n", onnx_output)
    print("Are outputs similar?", np.allclose(tflite_output, onnx_output, atol=1e-5))

# ========== Run everything ==========
tflite_path = create_tflite_model()
onnx_path = create_onnx_model()
compare_outputs(tflite_path, onnx_path)


Input:  [-1.  0.  1.]
Expected Output:
 [-0.6321206  0.         1.       ]
TFLite Output:
 [-0.63212055  0.          1.        ]
ONNX Output:
 [-0.63212055  0.          1.        ]
Are outputs similar? True


EXP

In [20]:
import tensorflow as tf
import numpy as np
import onnx
import onnxruntime as ort

# ========== Step 1: Create TFLite model with tf.raw_ops.Exp ==========
def create_tflite_model():
    class ExpModel(tf.Module):
        @tf.function(input_signature=[tf.TensorSpec(shape=[3], dtype=tf.float32)])
        def __call__(self, x):
            return tf.raw_ops.Exp(x=x)

    model = ExpModel()
    concrete_func = model.__call__.get_concrete_function()

    converter = tf.lite.TFLiteConverter.from_concrete_functions([concrete_func])
    tflite_model = converter.convert()

    with open("exp_model.tflite", "wb") as f:
        f.write(tflite_model)

    return "exp_model.tflite"

# ========== Step 2: Create ONNX model manually using Exp op ==========
def create_onnx_model():
    input_tensor = onnx.helper.make_tensor_value_info("input", onnx.TensorProto.FLOAT, [3])
    output_tensor = onnx.helper.make_tensor_value_info("output", onnx.TensorProto.FLOAT, [3])

    exp_node = onnx.helper.make_node(
        "Exp",
        inputs=["input"],
        outputs=["output"]
    )

    graph = onnx.helper.make_graph(
        nodes=[exp_node],
        name="ExpGraph",
        inputs=[input_tensor],
        outputs=[output_tensor]
    )

    model = onnx.helper.make_model(graph, producer_name="manual-exp-onnx")
    onnx.save(model, "exp_model.onnx")

    return "exp_model.onnx"

# ========== Step 3: Compare outputs ==========
def compare_outputs(tflite_model_path, onnx_model_path):
    input_data = np.array([-1.0, 0.0, 1.0], dtype=np.float32)
    expected_output = np.exp(input_data)

    # TFLite
    interpreter = tf.lite.Interpreter(model_path=tflite_model_path)
    interpreter.allocate_tensors()
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()

    interpreter.set_tensor(input_details[0]['index'], input_data)
    interpreter.invoke()
    tflite_output = interpreter.get_tensor(output_details[0]['index'])

    # ONNX
    session = ort.InferenceSession(onnx_model_path)
    onnx_output = session.run(None, {"input": input_data})[0]

    print("Input: ", input_data)
    print("Expected Output:\n", expected_output)
    print("TFLite Output:\n", tflite_output)
    print("ONNX Output:\n", onnx_output)
    print("Are outputs similar?", np.allclose(tflite_output, onnx_output, atol=1e-5))

# ========== Run Everything ==========
tflite_path = create_tflite_model()
onnx_path = create_onnx_model()
compare_outputs(tflite_path, onnx_path)


Input:  [-1.  0.  1.]
Expected Output:
 [0.36787942 1.         2.718282  ]
TFLite Output:
 [0.36787945 1.         2.7182817 ]
ONNX Output:
 [0.36787945 1.         2.7182817 ]
Are outputs similar? True


EXPAND_DIMS

In [23]:
import tensorflow as tf
import numpy as np
import onnx
import onnxruntime as ort

# ========= Step 1: Create TFLite model using tf.raw_ops.ExpandDims =========
def create_tflite_model():
    class ExpandDimsModel(tf.Module):
        @tf.function(input_signature=[
            tf.TensorSpec(shape=[3], dtype=tf.float32),
            tf.TensorSpec(shape=[], dtype=tf.int32)
        ])
        def __call__(self, input_tensor, axis):
            return tf.raw_ops.ExpandDims(input=input_tensor, axis=axis)

    model = ExpandDimsModel()
    concrete_func = model.__call__.get_concrete_function()

    converter = tf.lite.TFLiteConverter.from_concrete_functions([concrete_func])
    tflite_model = converter.convert()

    with open("expanddims_model.tflite", "wb") as f:
        f.write(tflite_model)

    return "expanddims_model.tflite"

# ========= Step 2: Create ONNX model using Unsqueeze op =========
def create_onnx_model():
    input_tensor = onnx.helper.make_tensor_value_info("input", onnx.TensorProto.FLOAT, [3])
    output_tensor = onnx.helper.make_tensor_value_info("output", onnx.TensorProto.FLOAT, [1, 3])

    axes_tensor = onnx.helper.make_tensor(name="axes", data_type=onnx.TensorProto.INT64, dims=[1], vals=[0])

    unsqueeze_node = onnx.helper.make_node(
        "Unsqueeze",
        inputs=["input", "axes"],
        outputs=["output"]
    )

    graph = onnx.helper.make_graph(
        nodes=[unsqueeze_node],
        name="ExpandDimsGraph",
        inputs=[input_tensor],
        outputs=[output_tensor],
        initializer=[axes_tensor]
    )

    model = onnx.helper.make_model(graph, producer_name="onnx-expanddims-model")
    onnx.save(model, "expanddims_model.onnx")

    return "expanddims_model.onnx"

# ========= Step 3: Compare outputs =========
def compare_outputs(tflite_model_path, onnx_model_path):
    input_data = np.array([1.0, 2.0, 3.0], dtype=np.float32)
    axis_data = np.array(0, dtype=np.int32)
    expected_output = np.expand_dims(input_data, axis=0)

    # Run TFLite
    interpreter = tf.lite.Interpreter(model_path=tflite_model_path)
    interpreter.allocate_tensors()
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()

    interpreter.set_tensor(input_details[0]['index'], input_data)
    interpreter.set_tensor(input_details[1]['index'], axis_data)
    interpreter.invoke()
    tflite_output = interpreter.get_tensor(output_details[0]['index'])

    # Run ONNX
    ort_session = ort.InferenceSession(onnx_model_path)
    onnx_output = ort_session.run(None, {"input": input_data})[0]

    print("Input:", input_data)
    print("Expected Output:\n", expected_output)
    print("TFLite Output:\n", tflite_output)
    print("ONNX Output:\n", onnx_output)
    print("Are outputs similar?", np.allclose(tflite_output, onnx_output, atol=1e-5))

# ========== Run Everything ==========
tflite_path = create_tflite_model()
onnx_path = create_onnx_model()
compare_outputs(tflite_path, onnx_path)


Input: [1. 2. 3.]
Expected Output:
 [[1. 2. 3.]]
TFLite Output:
 [[1. 2. 3.]]
ONNX Output:
 [[1. 2. 3.]]
Are outputs similar? True


Gather

In [25]:
import tensorflow as tf
import numpy as np
import onnx
import onnxruntime as ort

# ===== Step 1: Create TFLite model with tf.raw_ops.Gather =====
def create_tflite_model():
    class GatherModel(tf.Module):
        @tf.function(input_signature=[
            tf.TensorSpec(shape=[5], dtype=tf.float32),
            tf.TensorSpec(shape=[2], dtype=tf.int32),
        ])
        def __call__(self, params, indices):
            return tf.raw_ops.Gather(params=params, indices=indices)

    model = GatherModel()
    concrete_func = model.__call__.get_concrete_function()

    converter = tf.lite.TFLiteConverter.from_concrete_functions([concrete_func])
    tflite_model = converter.convert()

    with open("gather_model.tflite", "wb") as f:
        f.write(tflite_model)

    return "gather_model.tflite"

# ===== Step 2: Create ONNX model with Gather op =====
def create_onnx_model():
    input_tensor = onnx.helper.make_tensor_value_info("params", onnx.TensorProto.FLOAT, [5])
    indices_tensor = onnx.helper.make_tensor_value_info("indices", onnx.TensorProto.INT32, [2])
    output_tensor = onnx.helper.make_tensor_value_info("output", onnx.TensorProto.FLOAT, [2])

    gather_node = onnx.helper.make_node(
        "Gather",
        inputs=["params", "indices"],
        outputs=["output"],
        axis=0
    )

    graph = onnx.helper.make_graph(
        nodes=[gather_node],
        name="GatherGraph",
        inputs=[input_tensor, indices_tensor],
        outputs=[output_tensor]
    )

    model = onnx.helper.make_model(graph, producer_name="onnx-gather-model")
    onnx.save(model, "gather_model.onnx")
    return "gather_model.onnx"

# ===== Step 3: Compare outputs =====
def compare_outputs(tflite_model_path, onnx_model_path):
    params = np.array([10.0, 20.0, 30.0, 40.0, 50.0], dtype=np.float32)
    indices = np.array([1, 3], dtype=np.int32)
    expected_output = params[indices]

    # TFLite inference
    interpreter = tf.lite.Interpreter(model_path=tflite_model_path)
    interpreter.allocate_tensors()
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()

    interpreter.set_tensor(input_details[0]['index'], params)
    interpreter.set_tensor(input_details[1]['index'], indices)
    interpreter.invoke()
    tflite_output = interpreter.get_tensor(output_details[0]['index'])

    # ONNX inference
    ort_session = ort.InferenceSession(onnx_model_path)
    onnx_output = ort_session.run(None, {"params": params, "indices": indices})[0]

    # Compare
    print("Params:", params)
    print("Indices:", indices)
    print("Expected Output:", expected_output)
    print("TFLite Output:", tflite_output)
    print("ONNX Output:", onnx_output)
    print("Are outputs similar?", np.allclose(tflite_output, onnx_output, atol=1e-5))

# ===== Run All Steps =====
tflite_path = create_tflite_model()
onnx_path = create_onnx_model()
compare_outputs(tflite_path, onnx_path)


Params: [10. 20. 30. 40. 50.]
Indices: [1 3]
Expected Output: [20. 40.]
TFLite Output: [20. 40.]
ONNX Output: [20. 40.]
Are outputs similar? True


GatherND

In [28]:
import tensorflow as tf
import numpy as np
import onnx
import onnx.helper as helper
import onnxruntime as ort
from onnx import TensorProto, numpy_helper
import os

# ========== Step 1: Create TFLite Model ==========
def create_tflite_model():
    class GatherNdModel(tf.Module):
        @tf.function(input_signature=[
            tf.TensorSpec([2, 3], dtype=tf.float32),
            tf.TensorSpec([2, 2], dtype=tf.int32)
        ])
        def __call__(self, params, indices):
            return tf.raw_ops.GatherNd(params=params, indices=indices)

    model = GatherNdModel()
    concrete_func = model.__call__.get_concrete_function()

    converter = tf.lite.TFLiteConverter.from_concrete_functions([concrete_func])
    tflite_model = converter.convert()

    tflite_path = "gathernd_model.tflite"
    with open(tflite_path, "wb") as f:
        f.write(tflite_model)

    return tflite_path

# ========== Step 2: Create ONNX Model ==========
def create_onnx_model():
    input_params = helper.make_tensor_value_info("params", TensorProto.FLOAT, [2, 3])
    input_indices = helper.make_tensor_value_info("indices", TensorProto.INT64, [2, 2])
    output = helper.make_tensor_value_info("output", TensorProto.FLOAT, [2])

    node = helper.make_node(
        "GatherND",
        inputs=["params", "indices"],
        outputs=["output"]
    )

    graph = helper.make_graph(
        [node],
        "gathernd_graph",
        [input_params, input_indices],
        [output]
    )

    model = helper.make_model(graph)
    onnx.save(model, "gathernd_model.onnx")
    return "gathernd_model.onnx"

# ========== Step 3: Compare Outputs ==========
def compare_outputs(tflite_path, onnx_path):
    # Input Data
    params = np.array([[1, 2, 3], [4, 5, 6]], dtype=np.float32)
    tflite_indices = np.array([[0, 0], [1, 1]], dtype=np.int32)
    onnx_indices = np.array([[0, 0], [1, 1]], dtype=np.int64)

    # ---- TFLite Inference ----
    interpreter = tf.lite.Interpreter(model_path=tflite_path)
    interpreter.allocate_tensors()
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()

    interpreter.set_tensor(input_details[0]['index'], params)
    interpreter.set_tensor(input_details[1]['index'], tflite_indices)
    interpreter.invoke()
    tflite_output = interpreter.get_tensor(output_details[0]['index'])

    # ---- ONNX Inference ----
    sess = ort.InferenceSession(onnx_path)
    onnx_output = sess.run(None, {
        "params": params,
        "indices": onnx_indices
    })[0]

    # ---- Expected Output ----
    expected = np.array([1, 5], dtype=np.float32)

    print("Expected Output:\n", expected)
    print("TFLite Output:\n", tflite_output)
    print("ONNX Output:\n", onnx_output)
    print("Are outputs similar?", np.allclose(expected, tflite_output) and np.allclose(expected, onnx_output))

# ========== Run All Steps ==========
tflite_path = create_tflite_model()
onnx_path = create_onnx_model()
compare_outputs(tflite_path, onnx_path)


Expected Output:
 [1. 5.]
TFLite Output:
 [1. 5.]
ONNX Output:
 [1. 5.]
Are outputs similar? True


LeakyRELU

In [29]:
import tensorflow as tf
import numpy as np
import onnx
import onnx.helper as helper
import onnxruntime as ort
from onnx import TensorProto

# ========== Step 1: Create TFLite Model ==========
def create_tflite_model():
    class LeakyReluModel(tf.Module):
        @tf.function(input_signature=[
            tf.TensorSpec(shape=[3], dtype=tf.float32)
        ])
        def __call__(self, x):
            return tf.raw_ops.LeakyRelu(features=x, alpha=0.2)

    model = LeakyReluModel()
    concrete_func = model.__call__.get_concrete_function()

    converter = tf.lite.TFLiteConverter.from_concrete_functions([concrete_func])
    tflite_model = converter.convert()

    tflite_path = "leaky_relu_model.tflite"
    with open(tflite_path, "wb") as f:
        f.write(tflite_model)

    return tflite_path

# ========== Step 2: Create ONNX Model ==========
def create_onnx_model():
    input_tensor = helper.make_tensor_value_info("x", TensorProto.FLOAT, [3])
    output_tensor = helper.make_tensor_value_info("y", TensorProto.FLOAT, [3])

    leaky_relu_node = helper.make_node(
        "LeakyRelu",
        inputs=["x"],
        outputs=["y"],
        alpha=0.2
    )

    graph = helper.make_graph(
        [leaky_relu_node],
        "leaky_relu_graph",
        [input_tensor],
        [output_tensor]
    )

    model = helper.make_model(graph)
    onnx.save(model, "leaky_relu_model.onnx")
    return "leaky_relu_model.onnx"

# ========== Step 3: Compare Outputs ==========
def compare_outputs(tflite_path, onnx_path):
    input_data = np.array([-2.0, 0.0, 3.0], dtype=np.float32)

    # ---- TFLite Inference ----
    interpreter = tf.lite.Interpreter(model_path=tflite_path)
    interpreter.allocate_tensors()
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()

    interpreter.set_tensor(input_details[0]['index'], input_data)
    interpreter.invoke()
    tflite_output = interpreter.get_tensor(output_details[0]['index'])

    # ---- ONNX Inference ----
    sess = ort.InferenceSession(onnx_path)
    onnx_output = sess.run(None, {"x": input_data})[0]

    # ---- Expected ----
    expected_output = np.where(input_data > 0, input_data, input_data * 0.2)

    print("Input:\n", input_data)
    print("Expected Output:\n", expected_output)
    print("TFLite Output:\n", tflite_output)
    print("ONNX Output:\n", onnx_output)
    print("Are outputs similar?", np.allclose(tflite_output, onnx_output))

# ========== Run Everything ==========
tflite_path = create_tflite_model()
onnx_path = create_onnx_model()
compare_outputs(tflite_path, onnx_path)


Input:
 [-2.  0.  3.]
Expected Output:
 [-0.4  0.   3. ]
TFLite Output:
 [-0.4  0.   3. ]
ONNX Output:
 [-0.4  0.   3. ]
Are outputs similar? True


LoGISTIC

MAXPOOL_2D

In [30]:
import tensorflow as tf
import numpy as np
import onnx
import onnx.helper as helper
import onnxruntime as ort
from onnx import TensorProto

# ========== Step 1: Create TFLite Model ==========
def create_tflite_model():
    class MaxPoolModel(tf.Module):
        @tf.function(input_signature=[
            tf.TensorSpec(shape=[1, 4, 4, 1], dtype=tf.float32)
        ])
        def __call__(self, x):
            return tf.keras.layers.MaxPool2D(pool_size=(2, 2), strides=(2, 2), padding='VALID')(x)

    model = MaxPoolModel()
    concrete_func = model.__call__.get_concrete_function()

    converter = tf.lite.TFLiteConverter.from_concrete_functions([concrete_func])
    tflite_model = converter.convert()

    tflite_path = "maxpool2d_model.tflite"
    with open(tflite_path, "wb") as f:
        f.write(tflite_model)

    return tflite_path

# ========== Step 2: Create ONNX Model ==========
def create_onnx_model():
    input_tensor = helper.make_tensor_value_info("x", TensorProto.FLOAT, [1, 1, 4, 4])
    output_tensor = helper.make_tensor_value_info("y", TensorProto.FLOAT, [1, 1, 2, 2])

    node = helper.make_node(
        "MaxPool",
        inputs=["x"],
        outputs=["y"],
        kernel_shape=[2, 2],
        strides=[2, 2],
        pads=[0, 0, 0, 0]
    )

    graph = helper.make_graph(
        [node],
        "maxpool2d_graph",
        [input_tensor],
        [output_tensor]
    )

    model = helper.make_model(graph)
    onnx.save(model, "maxpool2d_model.onnx")
    return "maxpool2d_model.onnx"

# ========== Step 3: Compare Outputs ==========
def compare_outputs(tflite_path, onnx_path):
    input_data = np.array([[[[1.0], [2.0], [3.0], [4.0]],
                            [[5.0], [6.0], [7.0], [8.0]],
                            [[9.0], [10.0], [11.0], [12.0]],
                            [[13.0], [14.0], [15.0], [16.0]]]], dtype=np.float32)  # shape: (1, 4, 4, 1)

    # ---- TFLite Inference ----
    interpreter = tf.lite.Interpreter(model_path=tflite_path)
    interpreter.allocate_tensors()
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()

    interpreter.set_tensor(input_details[0]['index'], input_data)
    interpreter.invoke()
    tflite_output = interpreter.get_tensor(output_details[0]['index'])

    # ---- ONNX Inference ----
    nchw_input = np.transpose(input_data, (0, 3, 1, 2))  # (1, 1, 4, 4)
    sess = ort.InferenceSession(onnx_path)
    onnx_output = sess.run(None, {"x": nchw_input})[0]
    onnx_output_nhwc = np.transpose(onnx_output, (0, 2, 3, 1))

    print("Input:\n", input_data)
    print("TFLite Output:\n", tflite_output)
    print("ONNX Output:\n", onnx_output_nhwc)
    print("Are outputs similar?", np.allclose(tflite_output, onnx_output_nhwc))

# ========== Run Everything ==========
tflite_path = create_tflite_model()
onnx_path = create_onnx_model()
compare_outputs(tflite_path, onnx_path)


Input:
 [[[[ 1.]
   [ 2.]
   [ 3.]
   [ 4.]]

  [[ 5.]
   [ 6.]
   [ 7.]
   [ 8.]]

  [[ 9.]
   [10.]
   [11.]
   [12.]]

  [[13.]
   [14.]
   [15.]
   [16.]]]]
TFLite Output:
 [[[[ 6.]
   [ 8.]]

  [[14.]
   [16.]]]]
ONNX Output:
 [[[[ 6.]
   [ 8.]]

  [[14.]
   [16.]]]]
Are outputs similar? True


MAXIMUM

In [31]:
import tensorflow as tf
import numpy as np
import onnx
import onnx.helper as helper
import onnxruntime as ort
from onnx import TensorProto

# ========== Step 1: Create TFLite Model ==========
def create_tflite_model():
    class MaximumModel(tf.Module):
        @tf.function(input_signature=[
            tf.TensorSpec(shape=[2], dtype=tf.float32),
            tf.TensorSpec(shape=[2], dtype=tf.float32)
        ])
        def __call__(self, x, y):
            return tf.raw_ops.Maximum(x=x, y=y)

    model = MaximumModel()
    concrete_func = model.__call__.get_concrete_function()

    converter = tf.lite.TFLiteConverter.from_concrete_functions([concrete_func])
    tflite_model = converter.convert()

    tflite_path = "maximum_model.tflite"
    with open(tflite_path, "wb") as f:
        f.write(tflite_model)

    return tflite_path

# ========== Step 2: Create ONNX Model ==========
def create_onnx_model():
    input_x = helper.make_tensor_value_info("x", TensorProto.FLOAT, [2])
    input_y = helper.make_tensor_value_info("y", TensorProto.FLOAT, [2])
    output = helper.make_tensor_value_info("z", TensorProto.FLOAT, [2])

    node = helper.make_node(
        "Max",
        inputs=["x", "y"],
        outputs=["z"]
    )

    graph = helper.make_graph(
        [node],
        "maximum_graph",
        [input_x, input_y],
        [output]
    )

    model = helper.make_model(graph)
    onnx_path = "maximum_model.onnx"
    onnx.save(model, onnx_path)

    return onnx_path

# ========== Step 3: Compare Outputs ==========
def compare_outputs(tflite_path, onnx_path):
    input_x = np.array([1.0, 5.0], dtype=np.float32)
    input_y = np.array([4.0, 3.0], dtype=np.float32)

    # ---- TFLite Inference ----
    interpreter = tf.lite.Interpreter(model_path=tflite_path)
    interpreter.allocate_tensors()
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()

    interpreter.set_tensor(input_details[0]['index'], input_x)
    interpreter.set_tensor(input_details[1]['index'], input_y)
    interpreter.invoke()
    tflite_output = interpreter.get_tensor(output_details[0]['index'])

    # ---- ONNX Inference ----
    sess = ort.InferenceSession(onnx_path)
    onnx_output = sess.run(None, {"x": input_x, "y": input_y})[0]

    print("Input x:", input_x)
    print("Input y:", input_y)
    print("TFLite Output:", tflite_output)
    print("ONNX Output:", onnx_output)
    print("Are outputs similar?", np.allclose(tflite_output, onnx_output))

# ========== Run All ==========
tflite_path = create_tflite_model()
onnx_path = create_onnx_model()
compare_outputs(tflite_path, onnx_path)


Input x: [1. 5.]
Input y: [4. 3.]
TFLite Output: [4. 5.]
ONNX Output: [4. 5.]
Are outputs similar? True


MEAN

In [37]:
import tensorflow as tf
import numpy as np
import onnx
import onnx.helper as helper
import onnxruntime as ort
from onnx import TensorProto, OperatorSetIdProto

# ========== Step 1: Create TFLite Model ==========
def create_tflite_model():
    class MeanModel(tf.Module):
        @tf.function(input_signature=[
            tf.TensorSpec(shape=[2, 3], dtype=tf.float32),
            tf.TensorSpec(shape=[1], dtype=tf.int32),
        ])
        def __call__(self, input_tensor, axis_tensor):
            return tf.raw_ops.Mean(input=input_tensor, axis=axis_tensor, keep_dims=False)

    model = MeanModel()
    concrete_func = model.__call__.get_concrete_function()

    converter = tf.lite.TFLiteConverter.from_concrete_functions(
        [concrete_func],
        trackable_obj=model  # ✅ suppress deprecation warning
    )
    tflite_model = converter.convert()

    tflite_path = "mean_model.tflite"
    with open(tflite_path, "wb") as f:
        f.write(tflite_model)

    return tflite_path

# ========== Step 2: Create ONNX Model ==========
def create_onnx_model():
    input_tensor = helper.make_tensor_value_info("input", TensorProto.FLOAT, [2, 3])
    output_tensor = helper.make_tensor_value_info("output", TensorProto.FLOAT, [2])

    # Node with attribute axes set to [1]
    node = helper.make_node(
        "ReduceMean",
        inputs=["input"],
        outputs=["output"],
        keepdims=0,
        axes=[1]  # ✅ this works with opset >=13
    )

    graph = helper.make_graph(
        [node],
        "mean_graph",
        [input_tensor],
        [output_tensor]
    )

    opset = OperatorSetIdProto()
    opset.version = 13  # ✅ minimum required for axes to work correctly

    model = helper.make_model(graph, opset_imports=[opset])
    onnx_path = "mean_model.onnx"
    onnx.save(model, onnx_path)

    return onnx_path

# ========== Step 3: Compare Outputs ==========
def compare_outputs(tflite_path, onnx_path):
    input_data = np.array([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]], dtype=np.float32)
    axis_data = np.array([1], dtype=np.int32)

    # ---- TFLite Inference ----
    interpreter = tf.lite.Interpreter(model_path=tflite_path)
    interpreter.allocate_tensors()
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()

    # Set inputs by matching dtype
    for detail in input_details:
        if detail["dtype"] == np.float32:
            interpreter.set_tensor(detail['index'], input_data)
        elif detail["dtype"] == np.int32:
            interpreter.set_tensor(detail['index'], axis_data)

    interpreter.invoke()
    tflite_output = interpreter.get_tensor(output_details[0]['index'])

    # ---- ONNX Inference ----
    sess = ort.InferenceSession(onnx_path)
    onnx_output = sess.run(None, {"input": input_data})[0]

    print("Input:\n", input_data)
    print("Axis:", axis_data)
    print("TFLite Output:", tflite_output)
    print("ONNX Output:", onnx_output)
    print("Are outputs similar?", np.allclose(tflite_output, onnx_output))


# ========== Run All ==========
tflite_path = create_tflite_model()
onnx_path = create_onnx_model()
compare_outputs(tflite_path, onnx_path)


Input:
 [[1. 2. 3.]
 [4. 5. 6.]]
Axis: [1]
TFLite Output: [2. 5.]
ONNX Output: [2. 5.]
Are outputs similar? True


MINIMUM

In [38]:
import tensorflow as tf
import numpy as np
import onnx
import onnxruntime as ort
from onnx import helper, TensorProto

# ========= TFLite Model =========
class MinimumModel(tf.Module):
    @tf.function(input_signature=[
        tf.TensorSpec(shape=[2, 3], dtype=tf.float32),
        tf.TensorSpec(shape=[2, 3], dtype=tf.float32)
    ])
    def __call__(self, x, y):
        return tf.raw_ops.Minimum(x=x, y=y)

def create_tflite_model():
    model = MinimumModel()
    concrete_func = model.__call__.get_concrete_function()
    converter = tf.lite.TFLiteConverter.from_concrete_functions([concrete_func])
    tflite_model = converter.convert()
    with open("minimum_model.tflite", "wb") as f:
        f.write(tflite_model)
    return "minimum_model.tflite"

# ========= ONNX Model =========
def create_onnx_model():
    input1 = helper.make_tensor_value_info("x", TensorProto.FLOAT, [2, 3])
    input2 = helper.make_tensor_value_info("y", TensorProto.FLOAT, [2, 3])
    output = helper.make_tensor_value_info("output", TensorProto.FLOAT, [2, 3])

    node = helper.make_node(
        "Min",
        inputs=["x", "y"],
        outputs=["output"]
    )

    graph = helper.make_graph(
        [node],
        "MinimumGraph",
        inputs=[input1, input2],
        outputs=[output]
    )

    model = helper.make_model(graph, producer_name="minimum-test")
    onnx.save(model, "minimum_model.onnx")
    return "minimum_model.onnx"

# ========= Compare Outputs =========
def compare_outputs(tflite_path, onnx_path):
    x_data = np.array([[1.0, 7.0, 3.0], [4.0, 1.0, 8.0]], dtype=np.float32)
    y_data = np.array([[2.0, 5.0, 6.0], [3.0, 2.0, 7.0]], dtype=np.float32)

    # ---- TFLite Inference ----
    interpreter = tf.lite.Interpreter(model_path=tflite_path)
    interpreter.allocate_tensors()
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()

    interpreter.set_tensor(input_details[0]['index'], x_data)
    interpreter.set_tensor(input_details[1]['index'], y_data)
    interpreter.invoke()
    tflite_output = interpreter.get_tensor(output_details[0]['index'])

    # ---- ONNX Inference ----
    sess = ort.InferenceSession(onnx_path)
    onnx_output = sess.run(None, {"x": x_data, "y": y_data})[0]

    print("X:\n", x_data)
    print("Y:\n", y_data)
    print("TFLite Output:\n", tflite_output)
    print("ONNX Output:\n", onnx_output)
    print("Match:", np.allclose(tflite_output, onnx_output))

# ========= Run All =========
tflite_path = create_tflite_model()
onnx_path = create_onnx_model()
compare_outputs(tflite_path, onnx_path)


X:
 [[1. 7. 3.]
 [4. 1. 8.]]
Y:
 [[2. 5. 6.]
 [3. 2. 7.]]
TFLite Output:
 [[1. 5. 3.]
 [3. 1. 7.]]
ONNX Output:
 [[1. 5. 3.]
 [3. 1. 7.]]
Match: True


MUL

In [39]:
import tensorflow as tf
import numpy as np
import onnx
import onnxruntime as ort
from onnx import helper, TensorProto

# ========= TFLite Model =========
class MulModel(tf.Module):
    @tf.function(input_signature=[
        tf.TensorSpec(shape=[2, 3], dtype=tf.float32),
        tf.TensorSpec(shape=[2, 3], dtype=tf.float32)
    ])
    def __call__(self, x, y):
        return tf.raw_ops.Mul(x=x, y=y)

def create_tflite_model():
    model = MulModel()
    concrete_func = model.__call__.get_concrete_function()
    converter = tf.lite.TFLiteConverter.from_concrete_functions([concrete_func])
    tflite_model = converter.convert()
    with open("mul_model.tflite", "wb") as f:
        f.write(tflite_model)
    return "mul_model.tflite"

# ========= ONNX Model =========
def create_onnx_model():
    input1 = helper.make_tensor_value_info("x", TensorProto.FLOAT, [2, 3])
    input2 = helper.make_tensor_value_info("y", TensorProto.FLOAT, [2, 3])
    output = helper.make_tensor_value_info("output", TensorProto.FLOAT, [2, 3])

    node = helper.make_node(
        "Mul",
        inputs=["x", "y"],
        outputs=["output"]
    )

    graph = helper.make_graph(
        [node],
        "MulGraph",
        inputs=[input1, input2],
        outputs=[output]
    )

    model = helper.make_model(graph, producer_name="mul-test")
    onnx.save(model, "mul_model.onnx")
    return "mul_model.onnx"

# ========= Compare Outputs =========
def compare_outputs(tflite_path, onnx_path):
    x_data = np.array([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]], dtype=np.float32)
    y_data = np.array([[6.0, 5.0, 4.0], [3.0, 2.0, 1.0]], dtype=np.float32)

    # ---- TFLite Inference ----
    interpreter = tf.lite.Interpreter(model_path=tflite_path)
    interpreter.allocate_tensors()
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()

    interpreter.set_tensor(input_details[0]['index'], x_data)
    interpreter.set_tensor(input_details[1]['index'], y_data)
    interpreter.invoke()
    tflite_output = interpreter.get_tensor(output_details[0]['index'])

    # ---- ONNX Inference ----
    sess = ort.InferenceSession(onnx_path)
    onnx_output = sess.run(None, {"x": x_data, "y": y_data})[0]

    print("X:\n", x_data)
    print("Y:\n", y_data)
    print("TFLite Output:\n", tflite_output)
    print("ONNX Output:\n", onnx_output)
    print("Match:", np.allclose(tflite_output, onnx_output))

# ========= Run All =========
tflite_path = create_tflite_model()
onnx_path = create_onnx_model()
compare_outputs(tflite_path, onnx_path)


X:
 [[1. 2. 3.]
 [4. 5. 6.]]
Y:
 [[6. 5. 4.]
 [3. 2. 1.]]
TFLite Output:
 [[ 6. 10. 12.]
 [12. 10.  6.]]
ONNX Output:
 [[ 6. 10. 12.]
 [12. 10.  6.]]
Match: True


PACK

In [41]:
import tensorflow as tf
import numpy as np
import onnx
import onnxruntime as ort
from onnx import helper, TensorProto

# ========= TFLite Model =========
class PackModel(tf.Module):
    @tf.function(input_signature=[
        tf.TensorSpec(shape=[2], dtype=tf.float32),
        tf.TensorSpec(shape=[2], dtype=tf.float32),
        tf.TensorSpec(shape=[2], dtype=tf.float32)
    ])
    def __call__(self, x, y, z):
        return tf.raw_ops.Pack(values=[x, y, z], axis=0)  # Stacks into shape (3, 2)

def create_tflite_model():
    model = PackModel()
    concrete_func = model.__call__.get_concrete_function()
    converter = tf.lite.TFLiteConverter.from_concrete_functions([concrete_func])
    tflite_model = converter.convert()
    with open("pack_model.tflite", "wb") as f:
        f.write(tflite_model)
    return "pack_model.tflite"

def create_onnx_model():
    input1 = helper.make_tensor_value_info("x", TensorProto.FLOAT, [2])
    input2 = helper.make_tensor_value_info("y", TensorProto.FLOAT, [2])
    input3 = helper.make_tensor_value_info("z", TensorProto.FLOAT, [2])
    output = helper.make_tensor_value_info("output", TensorProto.FLOAT, [3, 2])

    # Axes for unsqueeze: [0]
    axes_const = helper.make_tensor("axes_const", TensorProto.INT64, [1], [0])

    # Unsqueeze nodes now take 2 inputs: data + axes
    unsq1 = helper.make_node("Unsqueeze", ["x", "axes_const"], ["x_unsq"])
    unsq2 = helper.make_node("Unsqueeze", ["y", "axes_const"], ["y_unsq"])
    unsq3 = helper.make_node("Unsqueeze", ["z", "axes_const"], ["z_unsq"])

    concat = helper.make_node(
        "Concat",
        inputs=["x_unsq", "y_unsq", "z_unsq"],
        outputs=["output"],
        axis=0
    )

    graph = helper.make_graph(
        [unsq1, unsq2, unsq3, concat],
        "PackGraph",
        inputs=[input1, input2, input3],
        outputs=[output],
        initializer=[axes_const]
    )

    model = helper.make_model(graph, producer_name="pack-test")
    onnx.save(model, "pack_model.onnx")
    return "pack_model.onnx"


# ========= Compare Outputs =========
def compare_outputs(tflite_path, onnx_path):
    x_data = np.array([1.0, 2.0], dtype=np.float32)
    y_data = np.array([3.0, 4.0], dtype=np.float32)
    z_data = np.array([5.0, 6.0], dtype=np.float32)

    # ---- TFLite Inference ----
    interpreter = tf.lite.Interpreter(model_path=tflite_path)
    interpreter.allocate_tensors()
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()

    interpreter.set_tensor(input_details[0]['index'], x_data)
    interpreter.set_tensor(input_details[1]['index'], y_data)
    interpreter.set_tensor(input_details[2]['index'], z_data)
    interpreter.invoke()
    tflite_output = interpreter.get_tensor(output_details[0]['index'])

    # ---- ONNX Inference ----
    sess = ort.InferenceSession(onnx_path)
    onnx_output = sess.run(None, {
        "x": x_data,
        "y": y_data,
        "z": z_data
    })[0]

    print("TFLite Output:\n", tflite_output)
    print("ONNX Output:\n", onnx_output)
    print("Match:", np.allclose(tflite_output, onnx_output))

# ========= Run All =========
tflite_path = create_tflite_model()
onnx_path = create_onnx_model()
compare_outputs(tflite_path, onnx_path)


TFLite Output:
 [[1. 2.]
 [3. 4.]
 [5. 6.]]
ONNX Output:
 [[1. 2.]
 [3. 4.]
 [5. 6.]]
Match: True


**PAD**

In [42]:
import tensorflow as tf
import numpy as np
import onnx
import onnxruntime as ort
from onnx import helper, TensorProto

# ==== 1. TF: Create TFLite model with tf.raw_ops.Pad ====
def create_tflite_model():
    class PadModel(tf.Module):
        @tf.function(input_signature=[
            tf.TensorSpec([2, 2], tf.float32, name="input"),
            tf.TensorSpec([2, 2], tf.int32, name="paddings")
        ])
        def __call__(self, x, paddings):
            return tf.raw_ops.Pad(input=x, paddings=paddings)

    model = PadModel()
    concrete_func = model.__call__.get_concrete_function()
    converter = tf.lite.TFLiteConverter.from_concrete_functions([concrete_func])
    tflite_model = converter.convert()

    with open("pad_model.tflite", "wb") as f:
        f.write(tflite_model)
    return "pad_model.tflite"

# ==== 2. ONNX: Create ONNX model with Pad ====
def create_onnx_model():
    input_tensor = helper.make_tensor_value_info("x", TensorProto.FLOAT, [2, 2])
    pads_tensor = helper.make_tensor_value_info("pads", TensorProto.INT64, [4])
    output_tensor = helper.make_tensor_value_info("output", TensorProto.FLOAT, [4, 4])

    pad_node = helper.make_node(
        "Pad",
        inputs=["x", "pads"],
        outputs=["output"],
        mode="constant"
    )

    graph = helper.make_graph(
        [pad_node],
        "PadGraph",
        inputs=[input_tensor, pads_tensor],
        outputs=[output_tensor]
    )

    model = helper.make_model(graph, producer_name="pad-test")
    onnx.save(model, "pad_model.onnx")
    return "pad_model.onnx"

# ==== 3. Compare outputs ====
def compare_outputs(tflite_path, onnx_path):
    input_data = np.array([[1, 2], [3, 4]], dtype=np.float32)
    paddings = np.array([[1, 1], [1, 1]], dtype=np.int32)
    onnx_pads = np.array([1, 1, 1, 1], dtype=np.int64)  # ONNX expects [top, left, bottom, right]

    # TFLite Inference
    interpreter = tf.lite.Interpreter(model_path=tflite_path)
    interpreter.allocate_tensors()
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()
    interpreter.set_tensor(input_details[0]['index'], input_data)
    interpreter.set_tensor(input_details[1]['index'], paddings)
    interpreter.invoke()
    tflite_output = interpreter.get_tensor(output_details[0]['index'])

    # ONNX Inference
    sess = ort.InferenceSession(onnx_path)
    onnx_output = sess.run(None, {"x": input_data, "pads": onnx_pads})[0]

    print("TFLite Output:\n", tflite_output)
    print("ONNX Output:\n", onnx_output)
    print("Match:", np.allclose(tflite_output, onnx_output))

# ==== Run ====
tflite_path = create_tflite_model()
onnx_path = create_onnx_model()
compare_outputs(tflite_path, onnx_path)


TFLite Output:
 [[0. 0. 0. 0.]
 [0. 1. 2. 0.]
 [0. 3. 4. 0.]
 [0. 0. 0. 0.]]
ONNX Output:
 [[0. 0. 0. 0.]
 [0. 1. 2. 0.]
 [0. 3. 4. 0.]
 [0. 0. 0. 0.]]
Match: True


POW

In [43]:
import tensorflow as tf
import numpy as np
import onnx
import onnxruntime as ort
from onnx import helper, TensorProto

# ==== 1. TF: Create TFLite model using tf.raw_ops.Pow ====
def create_tflite_model():
    class PowModel(tf.Module):
        @tf.function(input_signature=[
            tf.TensorSpec([2, 2], tf.float32, name="x"),
            tf.TensorSpec([2, 2], tf.float32, name="y")
        ])
        def __call__(self, x, y):
            return tf.raw_ops.Pow(x=x, y=y)

    model = PowModel()
    concrete_func = model.__call__.get_concrete_function()
    converter = tf.lite.TFLiteConverter.from_concrete_functions([concrete_func])
    tflite_model = converter.convert()

    with open("pow_model.tflite", "wb") as f:
        f.write(tflite_model)
    return "pow_model.tflite"

# ==== 2. ONNX: Create ONNX model using Pow ====
def create_onnx_model():
    x = helper.make_tensor_value_info("x", TensorProto.FLOAT, [2, 2])
    y = helper.make_tensor_value_info("y", TensorProto.FLOAT, [2, 2])
    output = helper.make_tensor_value_info("output", TensorProto.FLOAT, [2, 2])

    pow_node = helper.make_node(
        "Pow",
        inputs=["x", "y"],
        outputs=["output"]
    )

    graph = helper.make_graph(
        [pow_node],
        "PowGraph",
        inputs=[x, y],
        outputs=[output]
    )

    model = helper.make_model(graph, producer_name="pow-test")
    onnx.save(model, "pow_model.onnx")
    return "pow_model.onnx"

# ==== 3. Compare outputs ====
def compare_outputs(tflite_path, onnx_path):
    x_val = np.array([[2.0, 3.0], [4.0, 5.0]], dtype=np.float32)
    y_val = np.array([[3.0, 2.0], [0.5, 1.0]], dtype=np.float32)

    # TFLite Inference
    interpreter = tf.lite.Interpreter(model_path=tflite_path)
    interpreter.allocate_tensors()
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()
    interpreter.set_tensor(input_details[0]['index'], x_val)
    interpreter.set_tensor(input_details[1]['index'], y_val)
    interpreter.invoke()
    tflite_output = interpreter.get_tensor(output_details[0]['index'])

    # ONNX Inference
    sess = ort.InferenceSession(onnx_path)
    onnx_output = sess.run(None, {"x": x_val, "y": y_val})[0]

    print("TFLite Output:\n", tflite_output)
    print("ONNX Output:\n", onnx_output)
    print("Match:", np.allclose(tflite_output, onnx_output, rtol=1e-5, atol=1e-6))

# ==== Run ====
tflite_path = create_tflite_model()
onnx_path = create_onnx_model()
compare_outputs(tflite_path, onnx_path)


TFLite Output:
 [[8. 9.]
 [2. 5.]]
ONNX Output:
 [[8. 9.]
 [2. 5.]]
Match: True


PRELU

REDUCE_MAX

In [45]:
import tensorflow as tf
import numpy as np
import onnx
import onnxruntime as ort
from onnx import helper, TensorProto

# ==== 1. Create TFLite model with tf.math.reduce_max ====
def create_tflite_model():
    class ReduceMaxModel(tf.Module):
        @tf.function(input_signature=[
            tf.TensorSpec([2, 3], tf.float32, name="input")
        ])
        def __call__(self, x):
            return tf.math.reduce_max(x, axis=1, keepdims=False)

    model = ReduceMaxModel()
    concrete_func = model.__call__.get_concrete_function()
    converter = tf.lite.TFLiteConverter.from_concrete_functions([concrete_func])
    tflite_model = converter.convert()

    with open("reduce_max_model.tflite", "wb") as f:
        f.write(tflite_model)
    return "reduce_max_model.tflite"

# ==== 2. Create ONNX model using ReduceMax ====
def create_onnx_model():
    input_tensor = helper.make_tensor_value_info("input", TensorProto.FLOAT, [2, 3])
    output_tensor = helper.make_tensor_value_info("output", TensorProto.FLOAT, [2])

    reduce_node = helper.make_node(
        "ReduceMax",
        inputs=["input"],
        outputs=["output"],
        axes=[1],        # Pass axes as attribute for opset <= 12
        keepdims=0
    )

    graph = helper.make_graph(
        [reduce_node],
        "ReduceMaxGraph",
        inputs=[input_tensor],
        outputs=[output_tensor]
    )

    opset = onnx.helper.make_operatorsetid("", 11)  # 👈 Explicit opset 11
    model = helper.make_model(graph, opset_imports=[opset], producer_name="reduce_max_test")
    onnx.save(model, "reduce_max_model.onnx")
    return "reduce_max_model.onnx"

# ==== 3. Compare Outputs ====
def compare_outputs(tflite_path, onnx_path):
    input_val = np.array([[1.0, 5.0, 3.0],
                          [4.0, 2.0, 7.0]], dtype=np.float32)

    # TFLite Inference
    interpreter = tf.lite.Interpreter(model_path=tflite_path)
    interpreter.allocate_tensors()
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()
    interpreter.set_tensor(input_details[0]['index'], input_val)
    interpreter.invoke()
    tflite_output = interpreter.get_tensor(output_details[0]['index'])

    # ONNX Inference
    sess = ort.InferenceSession(onnx_path)
    onnx_output = sess.run(None, {"input": input_val})[0]

    print("TFLite Output:\n", tflite_output)
    print("ONNX Output:\n", onnx_output)
    print("Match:", np.allclose(tflite_output, onnx_output, rtol=1e-5, atol=1e-6))

# ==== Run ====
tflite_path = create_tflite_model()
onnx_path = create_onnx_model()
compare_outputs(tflite_path, onnx_path)


TFLite Output:
 [5. 7.]
ONNX Output:
 [5. 7.]
Match: True


REDUCE_PROD

In [47]:
import tensorflow as tf
import numpy as np
import onnx
import onnxruntime as ort
from onnx import helper, TensorProto

# Step 1: Create TFLite model with dynamic axis
def create_tflite_model():
    class ReduceProdModule(tf.Module):
        @tf.function(input_signature=[
            tf.TensorSpec(shape=[2, 3], dtype=tf.float32),
            tf.TensorSpec(shape=[1], dtype=tf.int32)
        ])
        def __call__(self, x, axis):
            return tf.math.reduce_prod(x, axis=axis)

    model = ReduceProdModule()
    concrete_func = model.__call__.get_concrete_function()
    converter = tf.lite.TFLiteConverter.from_concrete_functions([concrete_func])
    tflite_model = converter.convert()

    with open("reduce_prod_model.tflite", "wb") as f:
        f.write(tflite_model)

    return "reduce_prod_model.tflite"

# Step 2: Create ONNX model with hardcoded axis as attribute
def create_onnx_model():
    input_tensor = helper.make_tensor_value_info("input", TensorProto.FLOAT, [2, 3])
    output_tensor = helper.make_tensor_value_info("output", TensorProto.FLOAT, [2])

    reduce_node = helper.make_node(
        "ReduceProd",
        inputs=["input"],
        outputs=["output"],
        axes=[1],
        keepdims=0
    )

    graph = helper.make_graph(
        [reduce_node],
        "ReduceProdGraph",
        inputs=[input_tensor],
        outputs=[output_tensor]
    )

    model = helper.make_model(graph, opset_imports=[helper.make_operatorsetid("", 13)])
    onnx.save(model, "reduce_prod_model.onnx")
    return "reduce_prod_model.onnx"

# Step 3: Compare outputs
def compare_outputs(tflite_path, onnx_path):
    input_data = np.array([[1, 2, 3], [4, 5, 6]], dtype=np.float32)
    axis = np.array([1], dtype=np.int32)

    # TFLite inference
    interpreter = tf.lite.Interpreter(model_path=tflite_path)
    interpreter.allocate_tensors()
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()
    interpreter.set_tensor(input_details[0]['index'], input_data)
    interpreter.set_tensor(input_details[1]['index'], axis)
    interpreter.invoke()
    tflite_output = interpreter.get_tensor(output_details[0]['index'])

    # ONNX inference
    sess = ort.InferenceSession(onnx_path)
    onnx_output = sess.run(None, {"input": input_data})[0]

    # Compare
    print("TFLite Output:", tflite_output)
    print("ONNX Output:  ", onnx_output)
    print("Are outputs close? ->", np.allclose(tflite_output, onnx_output, atol=1e-5))

# Run all
if __name__ == "__main__":
    tflite_path = create_tflite_model()
    onnx_path = create_onnx_model()
    compare_outputs(tflite_path, onnx_path)


TFLite Output: [  6. 120.]
ONNX Output:   [  6. 120.]
Are outputs close? -> True


RELU

In [48]:
import tensorflow as tf
import numpy as np
import onnx
import onnxruntime as ort
from onnx import helper, TensorProto

# Step 1: Create TFLite model using tf.raw_ops.Relu
def create_tflite_model():
    class ReluModule(tf.Module):
        @tf.function(input_signature=[
            tf.TensorSpec(shape=[4], dtype=tf.float32)
        ])
        def __call__(self, x):
            return tf.raw_ops.Relu(features=x)

    model = ReluModule()
    concrete_func = model.__call__.get_concrete_function()
    converter = tf.lite.TFLiteConverter.from_concrete_functions([concrete_func])
    tflite_model = converter.convert()

    with open("relu_model.tflite", "wb") as f:
        f.write(tflite_model)

    return "relu_model.tflite"

# Step 2: Create ONNX model using Relu op
def create_onnx_model():
    input_tensor = helper.make_tensor_value_info("input", TensorProto.FLOAT, [4])
    output_tensor = helper.make_tensor_value_info("output", TensorProto.FLOAT, [4])

    relu_node = helper.make_node(
        "Relu",
        inputs=["input"],
        outputs=["output"]
    )

    graph = helper.make_graph(
        [relu_node],
        "ReluGraph",
        inputs=[input_tensor],
        outputs=[output_tensor]
    )

    model = helper.make_model(graph, opset_imports=[helper.make_operatorsetid("", 13)])
    onnx.save(model, "relu_model.onnx")
    return "relu_model.onnx"

# Step 3: Compare outputs from both models
def compare_outputs(tflite_path, onnx_path):
    input_data = np.array([-2.0, -1.0, 0.0, 3.0], dtype=np.float32)

    # TFLite inference
    interpreter = tf.lite.Interpreter(model_path=tflite_path)
    interpreter.allocate_tensors()
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()
    interpreter.set_tensor(input_details[0]['index'], input_data)
    interpreter.invoke()
    tflite_output = interpreter.get_tensor(output_details[0]['index'])

    # ONNX inference
    sess = ort.InferenceSession(onnx_path)
    onnx_output = sess.run(None, {"input": input_data})[0]

    # Compare
    print("Input:         ", input_data)
    print("TFLite Output: ", tflite_output)
    print("ONNX Output:   ", onnx_output)
    print("Are outputs close? ->", np.allclose(tflite_output, onnx_output, atol=1e-5))

# Run the pipeline
if __name__ == "__main__":
    tflite_path = create_tflite_model()
    onnx_path = create_onnx_model()
    compare_outputs(tflite_path, onnx_path)


Input:          [-2. -1.  0.  3.]
TFLite Output:  [0. 0. 0. 3.]
ONNX Output:    [0. 0. 0. 3.]
Are outputs close? -> True


Reshape

In [52]:
import tensorflow as tf
import numpy as np
import onnx
import onnxruntime as ort
import onnx.helper as helper
import onnx.numpy_helper as numpy_helper
import onnx.checker

# Suppress TensorFlow logs
tf.get_logger().setLevel("ERROR")

# --- Step 1: Create TFLite model with tf.raw_ops.Reshape
def create_tflite_model():
    class ReshapeModel(tf.Module):
        @tf.function(input_signature=[
            tf.TensorSpec([1, 6], tf.float32, name="input"),
            tf.TensorSpec([2], tf.int32, name="shape_tensor")
        ])
        def __call__(self, input, shape_tensor):
            return tf.raw_ops.Reshape(tensor=input, shape=shape_tensor)

    model = ReshapeModel()
    concrete_func = model.__call__.get_concrete_function()
    converter = tf.lite.TFLiteConverter.from_concrete_functions([concrete_func])
    tflite_model = converter.convert()

    tflite_path = "reshape_model.tflite"
    with open(tflite_path, "wb") as f:
        f.write(tflite_model)

    return tflite_path

# --- Step 2: Create ONNX model manually with equivalent Reshape op
def create_onnx_model():
    input_tensor = helper.make_tensor_value_info("input", onnx.TensorProto.FLOAT, [1, 6])
    shape_tensor = helper.make_tensor_value_info("shape", onnx.TensorProto.INT64, [2])
    output_tensor = helper.make_tensor_value_info("output", onnx.TensorProto.FLOAT, [2, 3])

    reshape_node = helper.make_node(
        "Reshape",
        inputs=["input", "shape"],
        outputs=["output"]
    )

    graph_def = helper.make_graph(
        nodes=[reshape_node],
        name="ReshapeGraph",
        inputs=[input_tensor, shape_tensor],
        outputs=[output_tensor]
    )

    model_def = helper.make_model(graph_def, producer_name="onnx-reshape")
    onnx.checker.check_model(model_def)

    onnx_path = "reshape_model.onnx"
    onnx.save(model_def, onnx_path)

    return onnx_path

# --- Step 3: Compare TFLite vs ONNX outputs
def compare_outputs(tflite_path, onnx_path):
    input_data = np.array([[1, 2, 3, 4, 5, 6]], dtype=np.float32)
    shape_data_tflite = np.array([2, 3], dtype=np.int32)
    shape_data_onnx = np.array([2, 3], dtype=np.int64)

    # Run TFLite
    interpreter = tf.lite.Interpreter(model_path=tflite_path)
    interpreter.allocate_tensors()
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()

    interpreter.set_tensor(input_details[0]['index'], input_data)
    interpreter.set_tensor(input_details[1]['index'], shape_data_tflite)
    interpreter.invoke()
    tflite_output = interpreter.get_tensor(output_details[0]['index'])

    # Run ONNX
    session = ort.InferenceSession(onnx_path)
    onnx_output = session.run(None, {
        "input": input_data,
        "shape": shape_data_onnx
    })[0]

    # Print and compare
    print("TFLite Output:\n", tflite_output)
    print("ONNX Output:\n", onnx_output)
    print("All Close:", np.allclose(tflite_output, onnx_output))


# --- Main
if __name__ == "__main__":
    tflite_path = create_tflite_model()
    onnx_path = create_onnx_model()
    compare_outputs(tflite_path, onnx_path)


TFLite Output:
 [[1. 2. 3.]
 [4. 5. 6.]]
ONNX Output:
 [[1. 2. 3.]
 [4. 5. 6.]]
All Close: True


Resize_bilinear

In [53]:
import tensorflow as tf
import numpy as np
import onnx
import onnx.helper as helper
import onnxruntime as ort
import onnx.numpy_helper as numpy_helper

# Disable TF logs
tf.get_logger().setLevel("ERROR")


# Step 1: Create TFLite model using tf.raw_ops.ResizeBilinear
def create_tflite_model():
    class ResizeBilinearModel(tf.Module):
        @tf.function(input_signature=[
            tf.TensorSpec(shape=[1, 2, 2, 1], dtype=tf.float32, name="input"),
            tf.TensorSpec(shape=[2], dtype=tf.int32, name="size")
        ])
        def __call__(self, input, size):
            return tf.raw_ops.ResizeBilinear(images=input, size=size, align_corners=False)

    model = ResizeBilinearModel()
    concrete_func = model.__call__.get_concrete_function()
    converter = tf.lite.TFLiteConverter.from_concrete_functions([concrete_func])
    tflite_model = converter.convert()

    tflite_path = "resize_bilinear_model.tflite"
    with open(tflite_path, "wb") as f:
        f.write(tflite_model)

    return tflite_path


# Step 2: Create ONNX model manually (Resize op with bilinear mode)
def create_onnx_model():
    input_tensor = helper.make_tensor_value_info("input", onnx.TensorProto.FLOAT, [1, 1, 2, 2])
    roi = helper.make_tensor(name="roi", data_type=onnx.TensorProto.FLOAT, dims=[0], vals=[])
    scales = numpy_helper.from_array(np.array([], dtype=np.float32), name="scales")  # not used
    sizes = numpy_helper.from_array(np.array([1, 1, 4, 4], dtype=np.int64), name="sizes")

    output_tensor = helper.make_tensor_value_info("output", onnx.TensorProto.FLOAT, [1, 1, 4, 4])

    resize_node = helper.make_node(
        "Resize",
        inputs=["input", "roi", "", "sizes"],  # scales is empty, sizes used
        outputs=["output"],
        mode="linear",
        coordinate_transformation_mode="asymmetric"
    )

    graph = helper.make_graph(
        [resize_node],
        "ResizeGraph",
        inputs=[input_tensor],
        outputs=[output_tensor],
        initializer=[roi, sizes]
    )

    model = helper.make_model(graph, producer_name="onnx-resize-bilinear")
    onnx.checker.check_model(model)

    onnx_path = "resize_bilinear_model.onnx"
    onnx.save(model, onnx_path)
    return onnx_path


# Step 3: Compare outputs
def compare_outputs(tflite_path, onnx_path):
    # Input shape: [1, 2, 2, 1] -> resize to [4, 4]
    input_data = np.array([[[[1], [2]], [[3], [4]]]], dtype=np.float32)
    size_data = np.array([4, 4], dtype=np.int32)  # height, width

    # TFLite
    interpreter = tf.lite.Interpreter(model_path=tflite_path)
    interpreter.allocate_tensors()
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()

    interpreter.set_tensor(input_details[0]["index"], input_data)
    interpreter.set_tensor(input_details[1]["index"], size_data)
    interpreter.invoke()
    tflite_output = interpreter.get_tensor(output_details[0]["index"])

    # ONNX: Transpose input to NCHW format
    input_nchw = input_data.transpose(0, 3, 1, 2)  # [1,1,2,2]
    ort_session = ort.InferenceSession(onnx_path)
    onnx_output = ort_session.run(None, {"input": input_nchw})[0]
    onnx_output_nhwc = onnx_output.transpose(0, 2, 3, 1)  # Back to NHWC

    print("TFLite output:\n", tflite_output)
    print("ONNX output:\n", onnx_output_nhwc)
    print("All Close:", np.allclose(tflite_output, onnx_output_nhwc, atol=1e-5))


# --- Main
if __name__ == "__main__":
    tflite_path = create_tflite_model()
    onnx_path = create_onnx_model()
    compare_outputs(tflite_path, onnx_path)


TFLite output:
 [[[[1. ]
   [1.5]
   [2. ]
   [2. ]]

  [[2. ]
   [2.5]
   [3. ]
   [3. ]]

  [[3. ]
   [3.5]
   [4. ]
   [4. ]]

  [[3. ]
   [3.5]
   [4. ]
   [4. ]]]]
ONNX output:
 [[[[1. ]
   [1.5]
   [2. ]
   [2. ]]

  [[2. ]
   [2.5]
   [3. ]
   [3. ]]

  [[3. ]
   [3.5]
   [4. ]
   [4. ]]

  [[3. ]
   [3.5]
   [4. ]
   [4. ]]]]
All Close: True


Resize nearest neighbor

In [54]:
import tensorflow as tf
import numpy as np
import onnx
import onnx.helper as helper
import onnx.numpy_helper as numpy_helper
import onnxruntime as ort

# Disable verbose TensorFlow logs
tf.get_logger().setLevel("ERROR")

# Step 1: Create a TFLite model using tf.raw_ops.ResizeNearestNeighbor
def create_tflite_model():
    class ResizeNearestModel(tf.Module):
        @tf.function(input_signature=[
            tf.TensorSpec(shape=[1, 2, 2, 1], dtype=tf.float32, name="input"),
            tf.TensorSpec(shape=[2], dtype=tf.int32, name="size")
        ])
        def __call__(self, input, size):
            return tf.raw_ops.ResizeNearestNeighbor(images=input, size=size, align_corners=False, half_pixel_centers=False)

    model = ResizeNearestModel()
    concrete_func = model.__call__.get_concrete_function()
    converter = tf.lite.TFLiteConverter.from_concrete_functions([concrete_func])
    tflite_model = converter.convert()

    tflite_path = "resize_nearest_model.tflite"
    with open(tflite_path, "wb") as f:
        f.write(tflite_model)

    return tflite_path


# Step 2: Create ONNX model manually using Resize (nearest mode)
def create_onnx_model():
    input_tensor = helper.make_tensor_value_info("input", onnx.TensorProto.FLOAT, [1, 1, 2, 2])
    roi = helper.make_tensor(name="roi", data_type=onnx.TensorProto.FLOAT, dims=[0], vals=[])
    sizes = numpy_helper.from_array(np.array([1, 1, 4, 4], dtype=np.int64), name="sizes")

    output_tensor = helper.make_tensor_value_info("output", onnx.TensorProto.FLOAT, [1, 1, 4, 4])

    resize_node = helper.make_node(
        "Resize",
        inputs=["input", "roi", "", "sizes"],
        outputs=["output"],
        mode="nearest",
        coordinate_transformation_mode="asymmetric",
        nearest_mode="floor"  # equivalent to half_pixel_centers=False
    )

    graph = helper.make_graph(
        [resize_node],
        "ResizeNearestGraph",
        inputs=[input_tensor],
        outputs=[output_tensor],
        initializer=[roi, sizes]
    )

    model = helper.make_model(graph, producer_name="onnx-resize-nearest")
    onnx.checker.check_model(model)

    onnx_path = "resize_nearest_model.onnx"
    onnx.save(model, onnx_path)
    return onnx_path


# Step 3: Compare outputs between TFLite and ONNX
def compare_outputs(tflite_path, onnx_path):
    input_data = np.array([[[[1], [2]], [[3], [4]]]], dtype=np.float32)  # shape: [1, 2, 2, 1]
    size_data = np.array([4, 4], dtype=np.int32)

    # TFLite inference
    interpreter = tf.lite.Interpreter(model_path=tflite_path)
    interpreter.allocate_tensors()
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()

    interpreter.set_tensor(input_details[0]["index"], input_data)
    interpreter.set_tensor(input_details[1]["index"], size_data)
    interpreter.invoke()
    tflite_output = interpreter.get_tensor(output_details[0]["index"])

    # ONNX inference (NCHW)
    input_nchw = input_data.transpose(0, 3, 1, 2)  # [1, 1, 2, 2]
    ort_session = ort.InferenceSession(onnx_path)
    onnx_output = ort_session.run(None, {"input": input_nchw})[0]
    onnx_output_nhwc = onnx_output.transpose(0, 2, 3, 1)  # Back to NHWC

    print("TFLite output:\n", tflite_output)
    print("ONNX output:\n", onnx_output_nhwc)
    print("Outputs match:", np.allclose(tflite_output, onnx_output_nhwc, atol=1e-5))


# --- Main Execution
if __name__ == "__main__":
    tflite_path = create_tflite_model()
    onnx_path = create_onnx_model()
    compare_outputs(tflite_path, onnx_path)


TFLite output:
 [[[[1.]
   [1.]
   [2.]
   [2.]]

  [[1.]
   [1.]
   [2.]
   [2.]]

  [[3.]
   [3.]
   [4.]
   [4.]]

  [[3.]
   [3.]
   [4.]
   [4.]]]]
ONNX output:
 [[[[1.]
   [1.]
   [2.]
   [2.]]

  [[1.]
   [1.]
   [2.]
   [2.]]

  [[3.]
   [3.]
   [4.]
   [4.]]

  [[3.]
   [3.]
   [4.]
   [4.]]]]
Outputs match: True


RSQRT

In [55]:
import tensorflow as tf
import numpy as np
import onnx
import onnx.helper as helper
import onnx.numpy_helper as numpy_helper
import onnxruntime as ort

# Step 1: Create a TFLite model using tf.raw_ops.Rsqrt
def create_tflite_model():
    class RsqrtModel(tf.Module):
        @tf.function(input_signature=[
            tf.TensorSpec(shape=[4], dtype=tf.float32, name="input")
        ])
        def __call__(self, input):
            return tf.raw_ops.Rsqrt(x=input)

    model = RsqrtModel()
    concrete_func = model.__call__.get_concrete_function()
    converter = tf.lite.TFLiteConverter.from_concrete_functions([concrete_func])
    tflite_model = converter.convert()

    tflite_path = "rsqrt_model.tflite"
    with open(tflite_path, "wb") as f:
        f.write(tflite_model)
    return tflite_path


# Step 2: Create an ONNX model using Pow with exponent -0.5
def create_onnx_model():
    input_tensor = helper.make_tensor_value_info("input", onnx.TensorProto.FLOAT, [4])
    output_tensor = helper.make_tensor_value_info("output", onnx.TensorProto.FLOAT, [4])

    exponent_tensor = numpy_helper.from_array(np.array([-0.5], dtype=np.float32), name="exponent")

    pow_node = helper.make_node(
        "Pow",
        inputs=["input", "exponent"],
        outputs=["output"]
    )

    graph = helper.make_graph(
        [pow_node],
        "RsqrtGraph",
        inputs=[input_tensor],
        outputs=[output_tensor],
        initializer=[exponent_tensor]
    )

    model = helper.make_model(graph, producer_name="onnx-rsqrt")
    onnx.checker.check_model(model)

    onnx_path = "rsqrt_model.onnx"
    onnx.save(model, onnx_path)
    return onnx_path


# Step 3: Compare outputs between TFLite and ONNX
def compare_outputs(tflite_path, onnx_path):
    input_data = np.array([1.0, 4.0, 9.0, 16.0], dtype=np.float32)

    # TFLite
    interpreter = tf.lite.Interpreter(model_path=tflite_path)
    interpreter.allocate_tensors()
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()
    interpreter.set_tensor(input_details[0]['index'], input_data)
    interpreter.invoke()
    tflite_output = interpreter.get_tensor(output_details[0]['index'])

    # ONNX
    ort_session = ort.InferenceSession(onnx_path)
    onnx_output = ort_session.run(None, {"input": input_data})[0]

    print("TFLite output:\n", tflite_output)
    print("ONNX output:\n", onnx_output)
    print("Outputs match:", np.allclose(tflite_output, onnx_output, atol=1e-5))


# --- Run Everything
if __name__ == "__main__":
    tflite_path = create_tflite_model()
    onnx_path = create_onnx_model()
    compare_outputs(tflite_path, onnx_path)


TFLite output:
 [0.99999994 0.49999997 0.3333333  0.24999999]
ONNX output:
 [1.         0.5        0.33333334 0.25      ]
Outputs match: True


Shape

In [58]:
import tensorflow as tf
import numpy as np
import onnx
import onnx.helper as helper
import onnxruntime as ort

# Step 1: Create a TFLite model using tf.raw_ops.Shape
def create_tflite_model():
    class ShapeModel(tf.Module):
        @tf.function(input_signature=[
            tf.TensorSpec(shape=[None, 3], dtype=tf.float32, name="input")
        ])
        def __call__(self, input):
            return tf.raw_ops.Shape(input=input)

    model = ShapeModel()
    concrete_func = model.__call__.get_concrete_function()
    converter = tf.lite.TFLiteConverter.from_concrete_functions([concrete_func])
    converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS]
    tflite_model = converter.convert()

    tflite_path = "shape_model.tflite"
    with open(tflite_path, "wb") as f:
        f.write(tflite_model)
    return tflite_path

# Step 2: Create the equivalent ONNX model using the Shape operator
def create_onnx_model():
    input_tensor = helper.make_tensor_value_info("input", onnx.TensorProto.FLOAT, [None, 3])
    output_tensor = helper.make_tensor_value_info("output", onnx.TensorProto.INT64, [None])

    shape_node = helper.make_node(
        "Shape",
        inputs=["input"],
        outputs=["output"]
    )

    graph = helper.make_graph(
        [shape_node],
        "ShapeGraph",
        inputs=[input_tensor],
        outputs=[output_tensor]
    )

    model = helper.make_model(graph, producer_name="onnx-shape")
    onnx.checker.check_model(model)

    onnx_path = "shape_model.onnx"
    onnx.save(model, onnx_path)
    return onnx_path

def compare_outputs(tflite_path, onnx_path):
    import numpy as np
    import tensorflow as tf
    import onnxruntime as ort

    # Load TFLite model and allocate tensors
    interpreter = tf.lite.Interpreter(model_path=tflite_path)
    interpreter.allocate_tensors()
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()

    # Get expected input shape and dtype
    expected_shape = input_details[0]['shape']
    input_dtype = input_details[0]['dtype']

    # Generate input matching the expected shape and dtype
    input_data = np.random.randn(*expected_shape).astype(input_dtype)

    # Set input tensor and run inference
    interpreter.set_tensor(input_details[0]['index'], input_data)
    interpreter.invoke()
    tflite_output = interpreter.get_tensor(output_details[0]['index'])

    # Run ONNX model
    ort_session = ort.InferenceSession(onnx_path)
    onnx_inputs = {ort_session.get_inputs()[0].name: input_data}
    onnx_output = ort_session.run(None, onnx_inputs)[0]

    print("TFLite output:\n", tflite_output)
    print("ONNX output:\n", onnx_output)
    print("Outputs match:", np.allclose(tflite_output, onnx_output, rtol=1e-05, atol=1e-06))



# --- Run Everything
if __name__ == "__main__":
    tflite_path = create_tflite_model()
    onnx_path = create_onnx_model()
    compare_outputs(tflite_path, onnx_path)


TFLite output:
 [1 3]
ONNX output:
 [1 3]
Outputs match: True


SLICE

In [59]:
import tensorflow as tf
import numpy as np
import onnx
import onnxruntime as ort
from onnx import helper, TensorProto
import tempfile
import os

# ===== Step 1: Create TF model with tf.raw_ops.Slice =====
class SliceModel(tf.Module):
    @tf.function(input_signature=[
        tf.TensorSpec([4, 4], tf.float32, name="input_tensor")
    ])
    def __call__(self, input_tensor):
        return tf.raw_ops.Slice(input=input_tensor,
                                begin=[1, 1],
                                size=[2, 2])

def create_tflite_model():
    model = SliceModel()
    concrete_func = model.__call__.get_concrete_function()

    converter = tf.lite.TFLiteConverter.from_concrete_functions([concrete_func])
    tflite_model = converter.convert()

    tflite_path = tempfile.mktemp(suffix=".tflite")
    with open(tflite_path, "wb") as f:
        f.write(tflite_model)

    return tflite_path

# ===== Step 2: Create ONNX model (manually) with Slice =====
def create_onnx_model():
    input_tensor = helper.make_tensor_value_info("input_tensor", TensorProto.FLOAT, [4, 4])
    output_tensor = helper.make_tensor_value_info("output_tensor", TensorProto.FLOAT, [2, 2])

    starts = helper.make_tensor("starts", TensorProto.INT64, [2], [1, 1])
    ends = helper.make_tensor("ends", TensorProto.INT64, [2], [3, 3])
    axes = helper.make_tensor("axes", TensorProto.INT64, [2], [0, 1])
    steps = helper.make_tensor("steps", TensorProto.INT64, [2], [1, 1])

    node = helper.make_node(
        "Slice",
        inputs=["input_tensor", "starts", "ends", "axes", "steps"],
        outputs=["output_tensor"]
    )

    graph = helper.make_graph(
        [node],
        "SliceGraph",
        [input_tensor],
        [output_tensor],
        initializer=[starts, ends, axes, steps]
    )

    model = helper.make_model(graph, producer_name="slice_model")
    onnx_path = tempfile.mktemp(suffix=".onnx")
    onnx.save(model, onnx_path)
    return onnx_path

# ===== Step 3: Compare outputs =====
def compare_outputs(tflite_path, onnx_path):
    interpreter = tf.lite.Interpreter(model_path=tflite_path)
    interpreter.allocate_tensors()
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()

    # Generate dummy input matching shape
    input_shape = input_details[0]['shape']
    input_data = np.random.rand(*input_shape).astype(np.float32)

    # TFLite Inference
    interpreter.set_tensor(input_details[0]['index'], input_data)
    interpreter.invoke()
    tflite_output = interpreter.get_tensor(output_details[0]['index'])

    # ONNX Inference
    ort_session = ort.InferenceSession(onnx_path)
    ort_inputs = {ort_session.get_inputs()[0].name: input_data}
    onnx_output = ort_session.run(None, ort_inputs)[0]

    print("Input:")
    print(input_data)
    print("\nTFLite Output:")
    print(tflite_output)
    print("\nONNX Output:")
    print(onnx_output)
    print("\nOutputs match:", np.allclose(tflite_output, onnx_output, rtol=1e-5, atol=1e-6))

# ===== Run All Steps =====
if __name__ == "__main__":
    tflite_path = create_tflite_model()
    onnx_path = create_onnx_model()
    compare_outputs(tflite_path, onnx_path)


Input:
[[0.8266229  0.7779769  0.9746376  0.38382208]
 [0.83923554 0.4286498  0.15540525 0.5513679 ]
 [0.48392966 0.3583903  0.00953653 0.611899  ]
 [0.7227892  0.39581096 0.38235706 0.99005204]]

TFLite Output:
[[0.4286498  0.15540525]
 [0.3583903  0.00953653]]

ONNX Output:
[[0.4286498  0.15540525]
 [0.3583903  0.00953653]]

Outputs match: True


SOFTMAX

In [60]:
import tensorflow as tf
import numpy as np
import tempfile
import onnx
import onnxruntime as ort
from onnx import helper, TensorProto

# ===== Step 1: Define TF model with tf.raw_ops.Softmax =====
class SoftmaxModel(tf.Module):
    @tf.function(input_signature=[
        tf.TensorSpec([1, 5], tf.float32, name="input_tensor")
    ])
    def __call__(self, input_tensor):
        return tf.raw_ops.Softmax(logits=input_tensor)

def create_tflite_model():
    model = SoftmaxModel()
    concrete_func = model.__call__.get_concrete_function()

    converter = tf.lite.TFLiteConverter.from_concrete_functions([concrete_func])
    tflite_model = converter.convert()

    tflite_path = tempfile.mktemp(suffix=".tflite")
    with open(tflite_path, "wb") as f:
        f.write(tflite_model)
    return tflite_path

# ===== Step 2: Manually create ONNX model with Softmax =====
def create_onnx_model():
    input_tensor = helper.make_tensor_value_info("input_tensor", TensorProto.FLOAT, [1, 5])
    output_tensor = helper.make_tensor_value_info("output_tensor", TensorProto.FLOAT, [1, 5])

    node = helper.make_node(
        "Softmax",
        inputs=["input_tensor"],
        outputs=["output_tensor"],
        axis=1  # same axis as TensorFlow
    )

    graph = helper.make_graph(
        [node],
        "SoftmaxGraph",
        [input_tensor],
        [output_tensor]
    )

    model = helper.make_model(graph, producer_name="softmax_model")
    onnx_path = tempfile.mktemp(suffix=".onnx")
    onnx.save(model, onnx_path)
    return onnx_path

# ===== Step 3: Compare outputs =====
def compare_outputs(tflite_path, onnx_path):
    interpreter = tf.lite.Interpreter(model_path=tflite_path)
    interpreter.allocate_tensors()
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()

    input_data = np.random.rand(*input_details[0]['shape']).astype(np.float32)

    # TFLite inference
    interpreter.set_tensor(input_details[0]['index'], input_data)
    interpreter.invoke()
    tflite_output = interpreter.get_tensor(output_details[0]['index'])

    # ONNX inference
    ort_session = ort.InferenceSession(onnx_path)
    ort_inputs = {ort_session.get_inputs()[0].name: input_data}
    onnx_output = ort_session.run(None, ort_inputs)[0]

    print("Input:")
    print(input_data)
    print("\nTFLite Output:")
    print(tflite_output)
    print("\nONNX Output:")
    print(onnx_output)
    print("\nOutputs match:", np.allclose(tflite_output, onnx_output, rtol=1e-5, atol=1e-6))

# ===== Run All Steps =====
if __name__ == "__main__":
    tflite_path = create_tflite_model()
    onnx_path = create_onnx_model()
    compare_outputs(tflite_path, onnx_path)


Input:
[[0.7104819  0.18546955 0.6870493  0.37917802 0.8519767 ]]

TFLite Output:
[[0.22530796 0.1332805  0.2200898  0.1617682  0.25955355]]

ONNX Output:
[[0.22530797 0.1332805  0.22008981 0.1617682  0.25955355]]

Outputs match: True


SPACE_TO_BATCHND

In [61]:
import tensorflow as tf
import numpy as np
import tempfile
import onnx
import onnxruntime as ort
from onnx import helper, TensorProto

# ========== TF Model with tf.raw_ops.SpaceToBatchND ==========
class STBNDModel(tf.Module):
    @tf.function(input_signature=[
        tf.TensorSpec([1, 2, 2, 1], tf.float32, name="input_tensor")
    ])
    def __call__(self, input_tensor):
        return tf.raw_ops.SpaceToBatchND(
            input=input_tensor,
            block_shape=tf.constant([2, 2], dtype=tf.int32),
            paddings=tf.constant([[0, 0], [0, 0]], dtype=tf.int32)
        )

def create_tflite_model():
    model = STBNDModel()
    concrete_func = model.__call__.get_concrete_function()

    converter = tf.lite.TFLiteConverter.from_concrete_functions([concrete_func])
    tflite_model = converter.convert()

    tflite_path = tempfile.mktemp(suffix=".tflite")
    with open(tflite_path, "wb") as f:
        f.write(tflite_model)
    return tflite_path

# ========== Equivalent ONNX Model ==========
def create_onnx_model():
    input_tensor = helper.make_tensor_value_info("input_tensor", TensorProto.FLOAT, [1, 2, 2, 1])
    output_tensor = helper.make_tensor_value_info("output_tensor", TensorProto.FLOAT, [4, 1, 1, 1])

    # Step 1: Reshape input from NHWC to NCHW for ONNX
    transpose_to_nchw = helper.make_node(
        "Transpose", inputs=["input_tensor"], outputs=["input_nchw"], perm=[0, 3, 1, 2]
    )

    # Step 2: Apply Pad (none here since paddings=[[0, 0], [0, 0]])
    # So this step is skipped

    # Step 3: SpaceToBatch logic using Reshape and Transpose
    # block_shape = [2, 2] → new batch = batch * prod(block_shape) = 1 * 4 = 4
    reshape_1 = helper.make_node(
        "Reshape",
        inputs=["input_nchw", "shape1"],
        outputs=["reshaped1"]
    )

    permute = helper.make_node(
        "Transpose", inputs=["reshaped1"], outputs=["transposed"], perm=[2, 4, 3, 5, 0, 1]
    )

    reshape_2 = helper.make_node(
        "Reshape",
        inputs=["transposed", "shape2"],
        outputs=["output_tensor"]
    )

    shape1_val = np.array([1, 1, 2, 2, 1, 1], dtype=np.int64)
    shape2_val = np.array([4, 1, 1, 1], dtype=np.int64)

    shape1_init = helper.make_tensor("shape1", TensorProto.INT64, shape1_val.shape, shape1_val)
    shape2_init = helper.make_tensor("shape2", TensorProto.INT64, shape2_val.shape, shape2_val)

    graph = helper.make_graph(
        [transpose_to_nchw, reshape_1, permute, reshape_2],
        "STBNDGraph",
        [input_tensor],
        [output_tensor],
        initializer=[shape1_init, shape2_init]
    )

    model = helper.make_model(graph, producer_name="space_to_batch_nd_model")
    onnx_path = tempfile.mktemp(suffix=".onnx")
    onnx.save(model, onnx_path)
    return onnx_path

# ========== Compare Outputs ==========
def compare_outputs(tflite_path, onnx_path):
    interpreter = tf.lite.Interpreter(model_path=tflite_path)
    interpreter.allocate_tensors()
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()

    input_data = np.array([[[[1.], [2.]], [[3.], [4.]]]], dtype=np.float32)

    interpreter.set_tensor(input_details[0]['index'], input_data)
    interpreter.invoke()
    tflite_output = interpreter.get_tensor(output_details[0]['index'])

    ort_session = ort.InferenceSession(onnx_path)
    ort_inputs = {ort_session.get_inputs()[0].name: input_data}
    onnx_output = ort_session.run(None, ort_inputs)[0]

    print("Input:\n", input_data)
    print("\nTFLite Output:\n", tflite_output)
    print("\nONNX Output:\n", onnx_output)
    print("\nOutputs Match:", np.allclose(tflite_output, onnx_output, rtol=1e-5, atol=1e-6))

# ========== Run ==========
if __name__ == "__main__":
    tflite_path = create_tflite_model()
    onnx_path = create_onnx_model()
    compare_outputs(tflite_path, onnx_path)


Input:
 [[[[1.]
   [2.]]

  [[3.]
   [4.]]]]

TFLite Output:
 [[[[1.]]]


 [[[2.]]]


 [[[3.]]]


 [[[4.]]]]

ONNX Output:
 [[[[1.]]]


 [[[2.]]]


 [[[3.]]]


 [[[4.]]]]

Outputs Match: True


SPACE_TO_DEPTH

In [63]:
import tensorflow as tf
import numpy as np
import tempfile
import onnx
import onnxruntime as ort
from onnx import helper, TensorProto

# ========= TF Model using tf.nn.space_to_depth =========
class SpaceToDepthModel(tf.Module):
    @tf.function(input_signature=[
        tf.TensorSpec([1, 2, 2, 4], tf.float32, name="input_tensor")
    ])
    def __call__(self, input_tensor):
        return tf.nn.space_to_depth(input_tensor, block_size=2)

def create_tflite_model():
    model = SpaceToDepthModel()
    concrete_func = model.__call__.get_concrete_function()

    converter = tf.lite.TFLiteConverter.from_concrete_functions([concrete_func])
    tflite_model = converter.convert()

    tflite_path = tempfile.mktemp(suffix=".tflite")
    with open(tflite_path, "wb") as f:
        f.write(tflite_model)
    return tflite_path

# ========= ONNX Model using SpaceToDepth =========
def create_onnx_model():
    input_tensor = helper.make_tensor_value_info("input_tensor", TensorProto.FLOAT, [1, 2, 2, 4])
    output_tensor = helper.make_tensor_value_info("output_tensor", TensorProto.FLOAT, [1, 1, 1, 16])

    # ONNX expects NCHW format for SpaceToDepth
    transpose1 = helper.make_node("Transpose", inputs=["input_tensor"], outputs=["nchw_input"], perm=[0, 3, 1, 2])

    space_to_depth = helper.make_node(
        "SpaceToDepth",
        inputs=["nchw_input"],
        outputs=["std_output"],
        blocksize=2
    )

    transpose2 = helper.make_node("Transpose", inputs=["std_output"], outputs=["output_tensor"], perm=[0, 2, 3, 1])

    graph = helper.make_graph(
        [transpose1, space_to_depth, transpose2],
        "SpaceToDepthGraph",
        [input_tensor],
        [output_tensor]
    )

    model = helper.make_model(graph, producer_name="space_to_depth_model")
    onnx_path = tempfile.mktemp(suffix=".onnx")
    onnx.save(model, onnx_path)
    return onnx_path

# ========= Compare Outputs =========
def compare_outputs(tflite_path, onnx_path):
    interpreter = tf.lite.Interpreter(model_path=tflite_path)
    interpreter.allocate_tensors()
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()

    # Create a dummy input [1, 2, 2, 4]
    input_data = np.array([[
        [[1, 2, 3, 4], [5, 6, 7, 8]],
        [[9, 10, 11, 12], [13, 14, 15, 16]]
    ]], dtype=np.float32)

    interpreter.set_tensor(input_details[0]['index'], input_data)
    interpreter.invoke()
    tflite_output = interpreter.get_tensor(output_details[0]['index'])

    ort_session = ort.InferenceSession(onnx_path)
    ort_inputs = {ort_session.get_inputs()[0].name: input_data}
    onnx_output = ort_session.run(None, ort_inputs)[0]

    print("Input:\n", input_data)
    print("\nTFLite Output:\n", tflite_output)
    print("\nONNX Output:\n", onnx_output)
    print("\nOutputs Match:", np.allclose(tflite_output, onnx_output, rtol=1e-5, atol=1e-6))

# ========= Run =========
if __name__ == "__main__":
    tflite_path = create_tflite_model()
    onnx_path = create_onnx_model()
    compare_outputs(tflite_path, onnx_path)


Input:
 [[[[ 1.  2.  3.  4.]
   [ 5.  6.  7.  8.]]

  [[ 9. 10. 11. 12.]
   [13. 14. 15. 16.]]]]

TFLite Output:
 [[[[ 1.  2.  3.  4.  5.  6.  7.  8.  9. 10. 11. 12. 13. 14. 15. 16.]]]]

ONNX Output:
 [[[[ 1.  2.  3.  4.  5.  6.  7.  8.  9. 10. 11. 12. 13. 14. 15. 16.]]]]

Outputs Match: True


SPLIT

In [66]:
import tensorflow as tf
import numpy as np
import tempfile
import onnx
import onnxruntime as ort
from onnx import helper, TensorProto, numpy_helper

# === TF Model ===
class SplitModel(tf.Module):
    @tf.function(input_signature=[
        tf.TensorSpec([6], tf.float32, name="input_tensor")
    ])
    def __call__(self, input_tensor):
        outputs = tf.raw_ops.Split(axis=0, num_split=3, value=input_tensor)
        return {
            "output_0": outputs[0],
            "output_1": outputs[1],
            "output_2": outputs[2],
        }

def create_tflite_model():
    model = SplitModel()
    concrete_func = model.__call__.get_concrete_function()
    converter = tf.lite.TFLiteConverter.from_concrete_functions([concrete_func])
    tflite_model = converter.convert()

    tflite_path = tempfile.mktemp(suffix=".tflite")
    with open(tflite_path, "wb") as f:
        f.write(tflite_model)
    return tflite_path

# === ONNX Model ===
def create_onnx_model():
    input_tensor = helper.make_tensor_value_info("input_tensor", TensorProto.FLOAT, [6])
    split_tensor = helper.make_tensor_value_info("split_tensor", TensorProto.INT64, [3])
    outputs = [
        helper.make_tensor_value_info("output_0", TensorProto.FLOAT, [2]),
        helper.make_tensor_value_info("output_1", TensorProto.FLOAT, [2]),
        helper.make_tensor_value_info("output_2", TensorProto.FLOAT, [2]),
    ]

    split_data = np.array([2, 2, 2], dtype=np.int64)
    split_const = helper.make_node(
        "Constant",
        inputs=[],
        outputs=["split_tensor"],
        value=numpy_helper.from_array(split_data, name="split_tensor")
    )

    split_node = helper.make_node(
        "Split",
        inputs=["input_tensor", "split_tensor"],
        outputs=["output_0", "output_1", "output_2"],
        axis=0
    )

    graph = helper.make_graph(
        [split_const, split_node],
        "SplitGraph",
        [input_tensor],
        outputs
    )

    model = helper.make_model(graph, producer_name="split_model")
    model.opset_import[0].version = 13  # Important!
    onnx_path = tempfile.mktemp(suffix=".onnx")
    onnx.save(model, onnx_path)
    return onnx_path

# === Compare ===
def compare_outputs(tflite_path, onnx_path):
    interpreter = tf.lite.Interpreter(model_path=tflite_path)
    interpreter.allocate_tensors()
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()

    input_data = np.array([1, 2, 3, 4, 5, 6], dtype=np.float32)
    interpreter.set_tensor(input_details[0]['index'], input_data)
    interpreter.invoke()
    tflite_outputs = [interpreter.get_tensor(o['index']) for o in output_details]

    ort_session = ort.InferenceSession(onnx_path)
    ort_outputs = ort_session.run(None, {"input_tensor": input_data})

    print("Input:\n", input_data)
    print("\nTFLite Outputs:")
    for i, out in enumerate(tflite_outputs):
        print(f"  output_{i}:\n{out}")

    print("\nONNX Outputs:")
    for i, out in enumerate(ort_outputs):
        print(f"  output_{i}:\n{out}")

    match = all(np.allclose(t, o, rtol=1e-5, atol=1e-6) for t, o in zip(tflite_outputs, ort_outputs))
    print("\n✅ Outputs Match:", match)

# === Run ===
if __name__ == "__main__":
    tflite_path = create_tflite_model()
    onnx_path = create_onnx_model()
    compare_outputs(tflite_path, onnx_path)


Input:
 [1. 2. 3. 4. 5. 6.]

TFLite Outputs:
  output_0:
[1. 2.]
  output_1:
[3. 4.]
  output_2:
[5. 6.]

ONNX Outputs:
  output_0:
[1. 2.]
  output_1:
[3. 4.]
  output_2:
[5. 6.]

✅ Outputs Match: True


SPLIT_V

In [72]:
import tensorflow as tf
import numpy as np
import onnx
import onnxruntime as ort
from onnx import helper, TensorProto
import tempfile

# ====================
# TF Model using tf.raw_ops.Split
# ====================
class SplitModel(tf.Module):
    @tf.function(input_signature=[tf.TensorSpec([6], tf.float32)])
    def __call__(self, x):
        return tf.raw_ops.Split(axis=0, value=x, num_split=3)

def create_tflite_model():
    model = SplitModel()
    concrete_func = model.__call__.get_concrete_function()
    converter = tf.lite.TFLiteConverter.from_concrete_functions([concrete_func])
    tflite_model = converter.convert()
    tflite_path = tempfile.mktemp(suffix=".tflite")
    with open(tflite_path, "wb") as f:
        f.write(tflite_model)
    return tflite_path

# ====================
# ONNX Model using Split
# ====================
def create_onnx_model():
    input_tensor = helper.make_tensor_value_info('input_tensor', TensorProto.FLOAT, [6])
    output_0 = helper.make_tensor_value_info('output_0', TensorProto.FLOAT, [2])
    output_1 = helper.make_tensor_value_info('output_1', TensorProto.FLOAT, [2])
    output_2 = helper.make_tensor_value_info('output_2', TensorProto.FLOAT, [2])

    node = helper.make_node(
        "Split",
        inputs=["input_tensor"],
        outputs=["output_0", "output_1", "output_2"],
        axis=0,
        split=[2, 2, 2]  # Make sure this line works only in opset 13+
    )

    graph = helper.make_graph(
        [node],
        "split_graph",
        [input_tensor],
        [output_0, output_1, output_2],
    )

    model = helper.make_model(
        graph,
        opset_imports=[helper.make_operatorsetid("", 13)]  # 👈 Critical
    )

    onnx_path = tempfile.mktemp(suffix=".onnx")
    onnx.save(model, onnx_path)
    return onnx_path

# ====================
# Compare Outputs
# ====================
def compare_outputs(tflite_path, onnx_path):
    input_data = np.array([1., 2., 3., 4., 5., 6.], dtype=np.float32)

    # Run TFLite
    interpreter = tf.lite.Interpreter(model_path=tflite_path)
    interpreter.allocate_tensors()
    input_index = interpreter.get_input_details()[0]["index"]
    interpreter.set_tensor(input_index, input_data)
    interpreter.invoke()
    tflite_outputs = [
        interpreter.get_tensor(output["index"])
        for output in interpreter.get_output_details()
    ]

    # Run ONNX
    sess = ort.InferenceSession(onnx_path)
    ort_outputs = sess.run(None, {"input_tensor": input_data})

    # Compare
    for i, (t_out, o_out) in enumerate(zip(tflite_outputs, ort_outputs)):
        print(f"Output {i}:")
        print("  TFLite:", t_out.flatten())
        print("  ONNX  :", o_out.flatten())
        print("  Close :", np.allclose(t_out, o_out))


# ====================
# Run Everything
# ====================
if __name__ == "__main__":
    tflite_path = create_tflite_model()
    onnx_path = create_onnx_model()
    compare_outputs(tflite_path, onnx_path)


InvalidGraph: [ONNXRuntimeError] : 10 : INVALID_GRAPH : Load model from /tmp/tmpu5_lmzl0.onnx failed:This is an invalid model. In Node, ("", Split, "", -1) : ("input_tensor": tensor(float),) -> ("output_0": tensor(float),"output_1": tensor(float),"output_2": tensor(float),) , Error Unrecognized attribute: split for operator Split

SQRT

In [73]:
import tensorflow as tf
import numpy as np
import onnx
import onnx.helper as helper
import onnx.numpy_helper as numpy_helper
import onnxruntime as ort
import tempfile

# Step 1: Define and save the TensorFlow model using tf.raw_ops.Sqrt
class SqrtModel(tf.Module):
    @tf.function(input_signature=[tf.TensorSpec(shape=[None], dtype=tf.float32)])
    def __call__(self, x):
        return {"sqrt": tf.raw_ops.Sqrt(x=x)}

def save_tf_model():
    model = SqrtModel()
    concrete_func = model.__call__.get_concrete_function()
    tf_path = tempfile.mkdtemp()
    tf.saved_model.save(model, tf_path, signatures=concrete_func)
    return tf_path

# Step 2: Manually create an ONNX model that replicates the Sqrt operation
def create_manual_onnx_model():
    input_tensor = helper.make_tensor_value_info("input", onnx.TensorProto.FLOAT, [None])
    output_tensor = helper.make_tensor_value_info("sqrt", onnx.TensorProto.FLOAT, [None])

    node = helper.make_node(
        "Sqrt",
        inputs=["input"],
        outputs=["sqrt"]
    )

    graph = helper.make_graph(
        nodes=[node],
        name="SqrtGraph",
        inputs=[input_tensor],
        outputs=[output_tensor]
    )

    model = helper.make_model(graph, opset_imports=[helper.make_opsetid("", 13)])
    onnx.checker.check_model(model)

    path = tempfile.mktemp(suffix=".onnx")
    onnx.save(model, path)
    return path

# Step 3: Compare ONNX model output with TensorFlow model output
def compare_outputs(tf_model_path, onnx_model_path):
    input_data = np.array([1.0, 4.0, 16.0, 100.0], dtype=np.float32)

    # TF output
    model = tf.saved_model.load(tf_model_path)
    tf_output = model.__call__(tf.constant(input_data))['sqrt'].numpy()
    print("TF Output:", tf_output)

    # ONNX output
    sess = ort.InferenceSession(onnx_model_path)
    onnx_output = sess.run(None, {"input": input_data})[0]
    print("ONNX Output:", onnx_output)

    assert np.allclose(tf_output, onnx_output, rtol=1e-4), "Mismatch between TF and ONNX outputs"

# Run everything
if __name__ == "__main__":
    tf_model_path = save_tf_model()
    onnx_model_path = create_manual_onnx_model()
    compare_outputs(tf_model_path, onnx_model_path)


TF Output: [ 1.  2.  4. 10.]
ONNX Output: [ 1.  2.  4. 10.]


SQUARED_DIFFERENCE

In [75]:
import tensorflow as tf
import numpy as np
import onnx
import onnx.helper as helper
import onnxruntime as ort
import tempfile

# Step 1: Define the TensorFlow model using tf.raw_ops.SquaredDifference
class SquaredDiffModel(tf.Module):
    @tf.function(input_signature=[
        tf.TensorSpec(shape=[None], dtype=tf.float32),
        tf.TensorSpec(shape=[None], dtype=tf.float32),
    ])
    def __call__(self, x, y):
        return {"squared_diff": tf.raw_ops.SquaredDifference(x=x, y=y)}

def save_tf_model():
    model = SquaredDiffModel()
    concrete_func = model.__call__.get_concrete_function()
    tf_path = tempfile.mkdtemp()
    tf.saved_model.save(model, tf_path, signatures=concrete_func)
    return tf_path

# Step 2: Manually create the ONNX model for SquaredDifference
def create_manual_onnx_model():
    input1 = helper.make_tensor_value_info("x", onnx.TensorProto.FLOAT, [None])
    input2 = helper.make_tensor_value_info("y", onnx.TensorProto.FLOAT, [None])
    output = helper.make_tensor_value_info("squared_diff", onnx.TensorProto.FLOAT, [None])

    # ONNX equivalent of SquaredDifference: (x - y) ** 2
    sub_node = helper.make_node("Sub", ["x", "y"], ["sub_out"])
    square_node = helper.make_node("Mul", ["sub_out", "sub_out"], ["squared_diff"])

    graph = helper.make_graph(
        nodes=[sub_node, square_node],
        name="SquaredDifferenceGraph",
        inputs=[input1, input2],
        outputs=[output],
    )

    model = helper.make_model(graph, opset_imports=[helper.make_opsetid("", 13)])
    onnx.checker.check_model(model)

    path = tempfile.mktemp(suffix=".onnx")
    onnx.save(model, path)
    return path

# Step 3: Compare outputs
def compare_outputs(tf_model_path, onnx_model_path):
    x_input = np.array([1.0, 2.0, 3.0], dtype=np.float32)
    y_input = np.array([0.0, 2.5, 1.0], dtype=np.float32)

    # TF output
    model = tf.saved_model.load(tf_model_path)
    tf_out = model.__call__(tf.constant(x_input), tf.constant(y_input))["squared_diff"].numpy()
    print("TF Output:", tf_out)

    # ONNX output
    sess = ort.InferenceSession(onnx_model_path)
    onnx_out = sess.run(None, {"x": x_input, "y": y_input})[0]
    print("ONNX Output:", onnx_out)

    assert np.allclose(tf_out, onnx_out, rtol=1e-4), "Mismatch between TF and ONNX outputs"

# === Run ===
if __name__ == "__main__":
    tf_model_path = save_tf_model()
    onnx_model_path = create_manual_onnx_model()
    compare_outputs(tf_model_path, onnx_model_path)


TF Output: [1.   0.25 4.  ]
ONNX Output: [1.   0.25 4.  ]


SQUEEZE

In [77]:
import tensorflow as tf
import numpy as np
import onnx
import onnx.helper as helper
import onnxruntime as ort
import tempfile

# Step 1: Define the TF model with tf.raw_ops.Squeeze
class SqueezeModel(tf.Module):
    @tf.function(input_signature=[
        tf.TensorSpec(shape=[1, 3, 1], dtype=tf.float32)
    ])
    def __call__(self, x):
        return {"squeezed": tf.raw_ops.Squeeze(input=x, axis=[0, 2])}

def save_tf_model():
    model = SqueezeModel()
    concrete_func = model.__call__.get_concrete_function()
    tf_path = tempfile.mkdtemp()
    tf.saved_model.save(model, tf_path, signatures=concrete_func)
    return tf_path

# Step 2: Create ONNX model equivalent to tf.raw_ops.Squeeze
def create_manual_onnx_model():
    import onnx.numpy_helper

    input_tensor = helper.make_tensor_value_info("x", onnx.TensorProto.FLOAT, [1, 3, 1])
    output_tensor = helper.make_tensor_value_info("squeezed", onnx.TensorProto.FLOAT, [3])

    # axes input as a constant tensor
    axes_initializer = onnx.numpy_helper.from_array(np.array([0, 2], dtype=np.int64), name="axes")

    squeeze_node = helper.make_node(
        "Squeeze",
        inputs=["x", "axes"],  # pass axes as input
        outputs=["squeezed"]
    )

    graph = helper.make_graph(
        [squeeze_node],
        "SqueezeGraph",
        [input_tensor],
        [output_tensor],
        initializer=[axes_initializer]
    )

    model = helper.make_model(graph, opset_imports=[helper.make_opsetid("", 13)])
    onnx.checker.check_model(model)

    path = tempfile.mktemp(suffix=".onnx")
    onnx.save(model, path)
    return path


# Step 3: Compare outputs
def compare_outputs(tf_model_path, onnx_model_path):
    x_input = np.array([[[1.], [2.], [3.]]], dtype=np.float32)  # shape [1, 3, 1]

    # TF output
    model = tf.saved_model.load(tf_model_path)
    tf_out = model.__call__(tf.constant(x_input))["squeezed"].numpy()
    print("TF Output:", tf_out)

    # ONNX output
    sess = ort.InferenceSession(onnx_model_path)
    onnx_out = sess.run(None, {"x": x_input})[0]
    print("ONNX Output:", onnx_out)

    assert np.allclose(tf_out, onnx_out), "Mismatch between TF and ONNX outputs"

# === Run ===
if __name__ == "__main__":
    tf_model_path = save_tf_model()
    onnx_model_path = create_manual_onnx_model()
    compare_outputs(tf_model_path, onnx_model_path)


TF Output: [1. 2. 3.]
ONNX Output: [1. 2. 3.]


STRIDED_SLICE

In [82]:
import numpy as np
import tensorflow as tf
import onnx
import onnxruntime as ort
from onnx import helper, TensorProto, numpy_helper
import tempfile
import os

# ========== TensorFlow Model ==========
def save_tf_model():
    class MyModule(tf.Module):
        @tf.function(input_signature=[tf.TensorSpec(shape=[5], dtype=tf.float32)])
        def __call__(self, x):
            return tf.raw_ops.StridedSlice(
                input=x,
                begin=[1],
                end=[2],
                strides=[1],
                begin_mask=0,
                end_mask=0,
                ellipsis_mask=0,
                new_axis_mask=0,
                shrink_axis_mask=1  # Removes axis to return scalar
            )

    model = MyModule()
    tf_path = tempfile.mkdtemp()
    tf.saved_model.save(model, tf_path)
    return tf_path

# ========== Manual ONNX Model ==========
def create_manual_onnx_model():
    input_tensor = helper.make_tensor_value_info('input_tensor', TensorProto.FLOAT, [5])
    output_tensor = helper.make_tensor_value_info('output', TensorProto.FLOAT, [])

    # Constants for Slice
    starts = numpy_helper.from_array(np.array([1], dtype=np.int64), name='starts')
    ends = numpy_helper.from_array(np.array([2], dtype=np.int64), name='ends')
    axes = numpy_helper.from_array(np.array([0], dtype=np.int64), name='axes')

    # Constant for Squeeze axes input
    squeeze_axes = numpy_helper.from_array(np.array([0], dtype=np.int64), name='squeeze_axes')

    slice_node = helper.make_node(
        'Slice',
        inputs=['input_tensor', 'starts', 'ends', 'axes'],
        outputs=['sliced_tensor']
    )

    squeeze_node = helper.make_node(
        'Squeeze',
        inputs=['sliced_tensor', 'squeeze_axes'],  # <-- axes as input not attribute
        outputs=['output']
    )

    graph = helper.make_graph(
        [slice_node, squeeze_node],
        'StridedSliceGraph',
        [input_tensor],
        [output_tensor],
        initializer=[starts, ends, axes, squeeze_axes]
    )

    model = helper.make_model(graph, producer_name='manual-stridedslice', opset_imports=[
        helper.make_opsetid("", 13)  # Ensure we're using opset 13+
    ])

    onnx.save(model, 'manual_stridedslice.onnx')
    return 'manual_stridedslice.onnx'


# ========== Compare Outputs ==========
def compare_outputs(tf_model_path, onnx_model_path):
    input_data = np.array([10, 20, 30, 40, 50], dtype=np.float32)

    # TF Output
    loaded = tf.saved_model.load(tf_model_path)
    tf_out = loaded.__call__(tf.constant(input_data)).numpy()
    print("TF Output:   ", tf_out)

    # ONNX Output
    sess = ort.InferenceSession(onnx_model_path)
    onnx_out = sess.run(None, {'input_tensor': input_data})[0]
    print("ONNX Output: ", onnx_out)

    # Match
    print("Match:", np.allclose(tf_out, onnx_out))

# ========== Run Everything ==========
if __name__ == "__main__":
    tf_path = save_tf_model()
    onnx_path = create_manual_onnx_model()
    compare_outputs(tf_path, onnx_path)


TF Output:    20.0
ONNX Output:  20.0
Match: True


SUB

In [83]:
import tensorflow as tf
import numpy as np
import onnx
from onnx import helper, TensorProto
import onnxruntime as ort
import os

# ========= 1. Create TensorFlow Model with tf.raw_ops.Sub =========
def save_tf_model():
    class SubModel(tf.Module):
        @tf.function(input_signature=[
            tf.TensorSpec(shape=[2], dtype=tf.float32),
            tf.TensorSpec(shape=[2], dtype=tf.float32),
        ])
        def __call__(self, x, y):
            return tf.raw_ops.Sub(x=x, y=y)

    model = SubModel()
    tf.saved_model.save(model, "tf_sub_model")
    return "tf_sub_model"

# ========= 2. Manually Create Equivalent ONNX Model =========
def create_manual_onnx_model():
    # Define inputs and output
    x = helper.make_tensor_value_info('x', TensorProto.FLOAT, [2])
    y = helper.make_tensor_value_info('y', TensorProto.FLOAT, [2])
    output = helper.make_tensor_value_info('output', TensorProto.FLOAT, [2])

    # Create Sub node
    sub_node = helper.make_node(
        'Sub',
        inputs=['x', 'y'],
        outputs=['output']
    )

    # Build graph
    graph = helper.make_graph(
        [sub_node],
        "SubGraph",
        inputs=[x, y],
        outputs=[output]
    )

    # Create model
    model = helper.make_model(graph, opset_imports=[helper.make_opsetid("", 13)])
    onnx.save(model, "manual_sub.onnx")
    return "manual_sub.onnx"

# ========= 3. Compare TF and ONNX Outputs =========
def compare_outputs(tf_path, onnx_path):
    # Input tensors
    x_val = np.array([10.0, 20.0], dtype=np.float32)
    y_val = np.array([1.0, 5.0], dtype=np.float32)

    # Run TF model
    tf_model = tf.saved_model.load(tf_path)
    tf_output = tf_model(x_val, y_val).numpy()
    print("TF Output:   ", tf_output)

    # Run ONNX model
    session = ort.InferenceSession(onnx_path, providers=["CPUExecutionProvider"])
    onnx_output = session.run(None, {"x": x_val, "y": y_val})[0]
    print("ONNX Output: ", onnx_output)

    # Compare
    print("Match:       ", np.allclose(tf_output, onnx_output, atol=1e-5))

# ========= 4. Run Everything =========
if __name__ == "__main__":
    tf_model_path = save_tf_model()
    onnx_model_path = create_manual_onnx_model()
    compare_outputs(tf_model_path, onnx_model_path)


TF Output:    [ 9. 15.]
ONNX Output:  [ 9. 15.]
Match:        True


SUM

In [84]:
import tensorflow as tf
import numpy as np
import onnx
from onnx import helper, TensorProto
import onnxruntime as ort

# ========= 1. TensorFlow Model using tf.raw_ops.Sum =========
def save_tf_model():
    class SumModel(tf.Module):
        @tf.function(input_signature=[
            tf.TensorSpec(shape=[2, 3], dtype=tf.float32)
        ])
        def __call__(self, x):
            return tf.raw_ops.Sum(input=x, axis=[1], keep_dims=False)

    model = SumModel()
    tf.saved_model.save(model, "tf_sum_model")
    return "tf_sum_model"

# ========= 2. Manual ONNX Model for ReduceSum =========
def create_manual_onnx_model():
    # Input & output
    input_tensor = helper.make_tensor_value_info('input', TensorProto.FLOAT, [2, 3])
    output_tensor = helper.make_tensor_value_info('output', TensorProto.FLOAT, [2])

    # Constant node for axes
    axes_initializer = helper.make_tensor(
        name="axes",
        data_type=TensorProto.INT64,
        dims=[1],
        vals=[1]
    )

    # ReduceSum Node
    reduce_sum_node = helper.make_node(
        "ReduceSum",
        inputs=["input", "axes"],
        outputs=["output"],
        keepdims=0  # Set to 0 for keep_dims=False
    )

    graph = helper.make_graph(
        [reduce_sum_node],
        "SumGraph",
        inputs=[input_tensor],
        outputs=[output_tensor],
        initializer=[axes_initializer]
    )

    model = helper.make_model(graph, opset_imports=[helper.make_opsetid("", 13)])
    onnx.save(model, "manual_sum.onnx")
    return "manual_sum.onnx"

# ========= 3. Compare Outputs =========
def compare_outputs(tf_model_path, onnx_model_path):
    x_val = np.array([[1, 2, 3], [4, 5, 6]], dtype=np.float32)

    # TF
    tf_model = tf.saved_model.load(tf_model_path)
    tf_output = tf_model(x_val).numpy()
    print("TF Output:   ", tf_output)

    # ONNX
    sess = ort.InferenceSession(onnx_model_path, providers=["CPUExecutionProvider"])
    onnx_output = sess.run(None, {"input": x_val})[0]
    print("ONNX Output: ", onnx_output)

    # Compare
    print("Match:       ", np.allclose(tf_output, onnx_output, atol=1e-5))

# ========= 4. Run All =========
if __name__ == "__main__":
    tf_path = save_tf_model()
    onnx_path = create_manual_onnx_model()
    compare_outputs(tf_path, onnx_path)


TF Output:    [ 6. 15.]
ONNX Output:  [ 6. 15.]
Match:        True


TANH

In [85]:
import tensorflow as tf
import numpy as np
import onnx
from onnx import helper, TensorProto
import onnxruntime as ort

# ========= 1. TensorFlow Model using tf.raw_ops.Tanh =========
def save_tf_model():
    class TanhModel(tf.Module):
        @tf.function(input_signature=[
            tf.TensorSpec(shape=[3], dtype=tf.float32)
        ])
        def __call__(self, x):
            return tf.raw_ops.Tanh(x=x)

    model = TanhModel()
    tf.saved_model.save(model, "tf_tanh_model")
    return "tf_tanh_model"

# ========= 2. Manual ONNX Model for Tanh =========
def create_manual_onnx_model():
    input_tensor = helper.make_tensor_value_info('input', TensorProto.FLOAT, [3])
    output_tensor = helper.make_tensor_value_info('output', TensorProto.FLOAT, [3])

    tanh_node = helper.make_node(
        "Tanh",
        inputs=["input"],
        outputs=["output"]
    )

    graph = helper.make_graph(
        [tanh_node],
        "TanhGraph",
        inputs=[input_tensor],
        outputs=[output_tensor]
    )

    model = helper.make_model(graph, opset_imports=[helper.make_opsetid("", 13)])
    onnx.save(model, "manual_tanh.onnx")
    return "manual_tanh.onnx"

# ========= 3. Compare Outputs =========
def compare_outputs(tf_model_path, onnx_model_path):
    x_val = np.array([0.0, 1.0, -1.0], dtype=np.float32)

    # TF
    tf_model = tf.saved_model.load(tf_model_path)
    tf_output = tf_model(x_val).numpy()
    print("TF Output:   ", tf_output)

    # ONNX
    sess = ort.InferenceSession(onnx_model_path, providers=["CPUExecutionProvider"])
    onnx_output = sess.run(None, {"input": x_val})[0]
    print("ONNX Output: ", onnx_output)

    print("Match:       ", np.allclose(tf_output, onnx_output, atol=1e-6))

# ========= 4. Run All =========
if __name__ == "__main__":
    tf_path = save_tf_model()
    onnx_path = create_manual_onnx_model()
    compare_outputs(tf_path, onnx_path)


TF Output:    [ 0.         0.7615942 -0.7615942]
ONNX Output:  [ 0.         0.7615942 -0.7615942]
Match:        True


Transpose

In [86]:
import tensorflow as tf
import numpy as np
import onnx
from onnx import helper, TensorProto
import onnxruntime as ort

# ========= 1. TensorFlow Model using tf.raw_ops.Transpose =========
def save_tf_model():
    class TransposeModel(tf.Module):
        @tf.function(input_signature=[
            tf.TensorSpec(shape=[2, 3], dtype=tf.float32)
        ])
        def __call__(self, x):
            # Explicitly define perm
            perm = tf.constant([1, 0])
            return tf.raw_ops.Transpose(x=x, perm=perm)

    model = TransposeModel()
    tf.saved_model.save(model, "tf_transpose_model")
    return "tf_transpose_model"

# ========= 2. Manual ONNX Model with Transpose =========
def create_manual_onnx_model():
    input_tensor = helper.make_tensor_value_info('input', TensorProto.FLOAT, [2, 3])
    output_tensor = helper.make_tensor_value_info('output', TensorProto.FLOAT, [3, 2])

    transpose_node = helper.make_node(
        "Transpose",
        inputs=["input"],
        outputs=["output"],
        perm=[1, 0]  # Explicit permutation
    )

    graph = helper.make_graph(
        [transpose_node],
        "TransposeGraph",
        inputs=[input_tensor],
        outputs=[output_tensor]
    )

    model = helper.make_model(graph, opset_imports=[helper.make_opsetid("", 13)])
    onnx.save(model, "manual_transpose.onnx")
    return "manual_transpose.onnx"

# ========= 3. Compare Outputs =========
def compare_outputs(tf_model_path, onnx_model_path):
    x_val = np.array([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]], dtype=np.float32)

    # TF
    tf_model = tf.saved_model.load(tf_model_path)
    tf_output = tf_model(x_val).numpy()
    print("TF Output:\n", tf_output)

    # ONNX
    sess = ort.InferenceSession(onnx_model_path, providers=["CPUExecutionProvider"])
    onnx_output = sess.run(None, {"input": x_val})[0]
    print("ONNX Output:\n", onnx_output)

    print("Match:", np.allclose(tf_output, onnx_output, atol=1e-6))

# ========= 4. Run All =========
if __name__ == "__main__":
    tf_path = save_tf_model()
    onnx_path = create_manual_onnx_model()
    compare_outputs(tf_path, onnx_path)


TF Output:
 [[1. 4.]
 [2. 5.]
 [3. 6.]]
ONNX Output:
 [[1. 4.]
 [2. 5.]
 [3. 6.]]
Match: True


Transpose_conv

Unideirectional_Sequence_lstm

Unpack

In [87]:
import tensorflow as tf
import numpy as np
import onnx
from onnx import helper, TensorProto
import onnxruntime as ort

# ========= 1. TensorFlow Model using tf.raw_ops.Unpack =========
def save_tf_model():
    class UnpackModel(tf.Module):
        @tf.function(input_signature=[
            tf.TensorSpec(shape=[3, 2], dtype=tf.float32)
        ])
        def __call__(self, x):
            outputs = tf.raw_ops.Unpack(value=x, num=3, axis=0)
            return {'out0': outputs[0], 'out1': outputs[1], 'out2': outputs[2]}

    model = UnpackModel()
    tf.saved_model.save(model, "tf_unpack_model")
    return "tf_unpack_model"

# ========= 2. Manual ONNX Model with Split operator =========
def create_manual_onnx_model():
    input_tensor = helper.make_tensor_value_info('input', TensorProto.FLOAT, [3, 2])
    out0 = helper.make_tensor_value_info('out0', TensorProto.FLOAT, [2])
    out1 = helper.make_tensor_value_info('out1', TensorProto.FLOAT, [2])
    out2 = helper.make_tensor_value_info('out2', TensorProto.FLOAT, [2])

    split_node = helper.make_node(
        "Split",
        inputs=["input"],
        outputs=["out0", "out1", "out2"],
        axis=0  # Split along axis 0
    )

    graph = helper.make_graph(
        [split_node],
        "UnpackGraph",
        inputs=[input_tensor],
        outputs=[out0, out1, out2]
    )

    model = helper.make_model(graph, opset_imports=[helper.make_opsetid("", 13)])
    onnx.save(model, "manual_unpack.onnx")
    return "manual_unpack.onnx"

# ========= 3. Compare Outputs =========
def compare_outputs(tf_model_path, onnx_model_path):
    x_val = np.array([[10., 20.], [30., 40.], [50., 60.]], dtype=np.float32)

    # TF
    tf_model = tf.saved_model.load(tf_model_path)
    tf_out = tf_model(x_val)
    tf_outputs = [tf_out['out0'].numpy(), tf_out['out1'].numpy(), tf_out['out2'].numpy()]
    print("TF Output:")
    for i, o in enumerate(tf_outputs):
        print(f"  out{i}: {o}")

    # ONNX
    sess = ort.InferenceSession(onnx_model_path, providers=["CPUExecutionProvider"])
    onnx_outputs = sess.run(None, {"input": x_val})
    print("\nONNX Output:")
    for i, o in enumerate(onnx_outputs):
        print(f"  out{i}: {o}")

    print("\nMatch:", all(np.allclose(tf_o, onnx_o, atol=1e-6) for tf_o, onnx_o in zip(tf_outputs, onnx_outputs)))

# ========= 4. Run All =========
if __name__ == "__main__":
    tf_path = save_tf_model()
    onnx_path = create_manual_onnx_model()
    compare_outputs(tf_path, onnx_path)


TF Output:
  out0: [10. 20.]
  out1: [30. 40.]
  out2: [50. 60.]

ONNX Output:
  out0: [[10. 20.]]
  out1: [[30. 40.]]
  out2: [[50. 60.]]

Match: True
